# Task 4: De-Corrupted 6-Backbone Foundation + Dual L2-SP Clean Fine-Tuning (Colab A100 Grandmaster Edition)

**100% Pure-Train Inductive Pipeline (Zero Test Leakage)** engineered for Google Colab **A100 (40/80 GB BF16)** within **~32–42 Compute Units (~2.5–3.2h)**, with automatic compatibility fallback for Colab L4/T4 and Kaggle `T4 × 2`:

1. **Zero-Upload Colab Fast-Path & Drive Persistence (`Cell 1 & Cell 2`):**
   - Auto-mounts Google Drive (`/content/drive/MyDrive/cpe342_t4_found_outputs`) for disconnect-proof checkpointing (`emb_v6_*.npz`, `folds_*/fold*.npz`, `t4_*.zip`).
   - Skips Kaggle/Drive walks on Colab -> downloads the official archive (~626 MB, ~30–60s) and enforces canonical Nov-12 `train.csv` (`PLY00001 = su7tkav.jpg`), `test.csv` (`ANS00001 = sjqyfwc.jpg`), and `folds_sgkf.npy`.
2. **Deterministic De-Corruption v4 (`Cell 3`):**
   - `BoxBlur(2)` + Theil-Sen median slope ($R^2 \ge 0.85$) derotation & conservative crop, `YCbCr` chroma denoise (`GaussianBlur(1.5)` on `Cb/Cr`, `MedianFilter(3)` on `Y`), and capped ($\le 2.2\times$) contrast restoration.
3. **Stage 1: 6 Giant/Large Open-World Backbones × 6-View Extraction + CL2N Heads (`Cell 4 & Cell 5`):**
   - **6 Diverse Backbones** @ `(224, 448)` on A100/H100 (or `(196, 392)`/`(192, 384)` on T4/L4):
     1. `siglip2_so400m` (SigLIP2-SO400M / SigLIP-SO400M Vision-Language)
     2. `dinov3_vitl` (Meta DINOv3-ViT-Large SSL)
     3. `dinov2_vitg14` (Meta DINOv2-ViT-Giant/14 Reg4 SSL)
     4. `eva02_large` (BAAI EVA02-Large MIM)
     5. `clip_cnxl` (OpenCLIP LAION-2B ConvNeXt-XXLarge/Large)
     6. `clip_vith14` (OpenCLIP DFN5B ViT-Huge/14 or CLIP ViT-L/14@336)
   - **6-View GPU Extraction:** `[raw, hflip(raw), decorrupt_v4, hflip(decorrupt_v4), crop82(raw), crop82(decorrupt_v4)]`.
   - **Fold-Internal CL2N Centering + 3 Upgraded Heads:** `LogReg-CL2N`, 5-seed `CosineResMLP` (`hidden=768`, `block_drop=0.22`, `Beta(0.4, 0.4)` Mixup, `tau=1.0` Logit-Adj CE), and **Nested `PerBackbone-SLSQP`**, exported immediately as `t4_foundvlm.zip`.
4. **Stage 2 (`RUN_STAGE2_CLEAN_FINETUNE = "auto"` -> ON for A100/H100): Dual L2-SP Clean Fine-Tuning (`Cell 6`):**
   - Fine-tunes **`clean_siglip`** (`vit_so400m_patch14_siglip_384`) AND **`clean_dinov3l`** (`vit_large_patch16_dinov3.lvd1689m`) @ `(224, 448)` with **`P_NIGHTMARE = 0.0`** (50% raw + 50% `decorrupt_v4`), **L2-SP** anchoring toward pretrained $\theta_0$ (`SP_WEIGHT_DECAY = 1.5e-2`), `tau = 1.0` Logit-Adj CE, EMA `0.996`, and **6-View GPU TTA**.

> Outputs (`t4_foundvlm.zip`, `t4_clean_siglip.zip`, `t4_clean_dinov3l.zip`, `t4_found_bundles.zip`) are auto-discovered by `fuse_t4_sota_trio.py`!

In [ ]:
# Cell 1: Dependencies (timm >= 1.0.22), HF Token Guard, Colab Drive Auto-Mount & GPU Status Banner
import os
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
import importlib, importlib.metadata, shutil, subprocess, sys

def parse_ver(v: str):
    parts = []
    for tok in v.split(".")[:3]:
        digits = "".join(ch for ch in tok if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    return tuple(parts)

need_install = []
try:
    cur_timm = importlib.metadata.version("timm")
    if parse_ver(cur_timm) < (1, 0, 22):
        print(f"Upgrading timm {cur_timm} -> >=1.0.22 (required for DINOv3, SigLIP2 & DINOv2-Giant)...")
        need_install.append("timm>=1.0.22")
except importlib.metadata.PackageNotFoundError:
    need_install.append("timm>=1.0.22")

for pkg in ("scikit-learn", "pandas", "numpy", "pillow"):
    try:
        importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        need_install.append(pkg)

if need_install:
    had_stale_timm = "timm" in sys.modules
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", *need_install])
    importlib.invalidate_caches()
    if had_stale_timm:
        raise RuntimeError("Upgraded timm in-place; click Restart Session once and Run All!")

import numpy as np
import pandas as pd
import timm
import torch

print(f"Verified versions -> timm=={timm.__version__} | torch=={torch.__version__} | python=={sys.version.split()[0]}")
assert parse_ver(timm.__version__) >= (1, 0, 22), f"Restart session once! Loaded stale timm {timm.__version__}"

# --- Run Configuration ---
RUN_STAGE1_FOUNDATION_PROBES = True          # Stage 1: 6 Foundation Backbones x 6-View + CL2N Heads (~18-30m on A100)
RUN_STAGE2_CLEAN_FINETUNE = "auto"           # "auto" -> True on A100/H100 (VRAM >= 36GB), False on L4/T4; or True/False
STAGE2_MODEL_KEYS = ["clean_siglip", "clean_dinov3l"]
STAGE2_EPOCHS = 4
STAGE2_FOLDS_TO_RUN = None                   # None -> all 5 folds
SP_WEIGHT_DECAY = 1.5e-2                     # L2-SP regularization anchoring unfrozen backbone toward pretrained weights
USE_COLAB_DRIVE = True                       # Persist cache & zips in Google Drive on Colab (disconnect-proof!)
COLAB_DRIVE_DIR = "/content/drive/MyDrive/cpe342_t4_found_outputs"
ENABLE_DUAL_GPU_PARALLEL = True
DISABLE_GRAD_CKPT_HIGH_VRAM = True
SEED = 42

def select_amp_dtype(device: torch.device):
    """Use bfloat16 strictly on Ampere+ (cc >= 8.0); Turing T4 (sm_75) uses float16."""
    if device.type != "cuda":
        return None, False
    dev_idx = 0 if device.index is None else int(device.index)
    cc_major, _ = torch.cuda.get_device_capability(dev_idx)
    if cc_major >= 8 and torch.cuda.is_bf16_supported():
        return torch.bfloat16, False
    return torch.float16, True

def resolve_a100_config(vram_gb: float, run_stage2_cfg="auto"):
    high_res = (vram_gb >= 36.0)
    high_vram_no_ckpt = (vram_gb >= 36.0)
    h100_turbo = (vram_gb >= 60.0)
    fast_t4 = (vram_gb < 20.0)
    n_views = 4 if fast_t4 else 6
    bs_mult = 5.0 if vram_gb >= 70.0 else (4.0 if vram_gb >= 36.0 else (2.5 if vram_gb >= 20.0 else 2.0))
    do_stage2 = (vram_gb >= 36.0) if str(run_stage2_cfg).lower() == "auto" else bool(run_stage2_cfg)
    return high_res, high_vram_no_ckpt, h100_turbo, bs_mult, do_stage2, fast_t4, n_views

IS_KAGGLE = ("KAGGLE_KERNEL_RUN_TYPE" in os.environ) or (os.path.isdir("/kaggle/working") and not os.path.isdir("/content"))
IS_COLAB = (not IS_KAGGLE) and (("COLAB_RELEASE_TAG" in os.environ) or ("COLAB_GPU" in os.environ) or os.path.isdir("/content"))
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
AMP_DTYPE, USE_SCALER = select_amp_dtype(DEVICE)
N_GPUS = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
NUM_WORKERS = min(8, max(2, (os.cpu_count() or 2) - 1)) if DEVICE.type == "cuda" else 0

GPU_VRAM_GB = 0.0
GPU_NAME = str(DEVICE)
if DEVICE.type == "cuda":
    cc_major, _ = torch.cuda.get_device_capability(0)
    torch.backends.cudnn.benchmark = (cc_major >= 8)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    GPU_NAME = torch.cuda.get_device_properties(0).name
    for gi in range(N_GPUS):
        gp = torch.cuda.get_device_properties(gi)
        gvram = gp.total_memory / (1024**3)
        GPU_VRAM_GB = max(GPU_VRAM_GB, gvram)
        g_amp, _ = select_amp_dtype(torch.device(f"cuda:{gi}"))
        print(f"GPU[{gi}]: {gp.name} ({gvram:.1f} GB VRAM) | cc={gp.major}.{gp.minor} | amp={g_amp} | cpus={os.cpu_count()}")
else:
    print(f"Running on {DEVICE} (no CUDA GPU detected)")

HIGH_RES_A100, HIGH_VRAM_NO_CKPT, H100_TURBO, EXTRACT_BS_MULT, DO_STAGE2, FAST_T4_TIER, N_VIEWS = resolve_a100_config(GPU_VRAM_GB, RUN_STAGE2_CLEAN_FINETUNE)

if IS_KAGGLE:
    OUT_DIR = "/kaggle/working/t4_found_outputs"
elif IS_COLAB:
    OUT_DIR = "/content/t4_found_outputs"
    if USE_COLAB_DRIVE:
        if not os.path.isdir("/content/drive/MyDrive"):
            try:
                from google.colab import drive
                drive.mount("/content/drive", force_remount=False)
            except Exception as _exc:
                print(f"[Colab Drive Warning] Mount skipped ({_exc}); using local {OUT_DIR}")
        if os.path.isdir("/content/drive/MyDrive"):
            OUT_DIR = COLAB_DRIVE_DIR
            print(f"[Colab Drive OK] Persisting outputs -> {OUT_DIR}")
        else:
            print(f"[Colab Local] Drive unavailable; using local {OUT_DIR} (remember to download before disconnect!)")
else:
    OUT_DIR = os.path.abspath("./t4_found_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# Import any cached Stage-1 embeddings or Stage-2 fold checkpoints attached via /kaggle/input or /content
_import_roots = []
if IS_KAGGLE and os.path.isdir("/kaggle/input"):
    _import_roots.append("/kaggle/input")
elif IS_COLAB and OUT_DIR != "/content/t4_found_outputs":
    _import_roots.append("/content")

for _ir in _import_roots:
    _base_d = _ir.rstrip(os.sep).count(os.sep)
    for root, dirs, files in os.walk(_ir, followlinks=True):
        dirs[:] = [d for d in dirs if d not in ("train", "test", "t4_data", "t4_official_raw", "drive", "__pycache__", ".git")]
        if root.rstrip(os.sep).count(os.sep) - _base_d >= 4:
            dirs[:] = []
        bname = os.path.basename(root)
        if bname.startswith("folds_"):
            dst_fd = os.path.join(OUT_DIR, bname)
            os.makedirs(dst_fd, exist_ok=True)
            for fn in files:
                if fn.startswith("fold") and fn.endswith(".npz"):
                    dst_p = os.path.join(dst_fd, fn)
                    if not os.path.isfile(dst_p):
                        shutil.copy2(os.path.join(root, fn), dst_p)
                        print(f"Imported Stage-2 fold checkpoint: {bname}/{fn}")
        for fn in files:
            if fn.startswith("emb_") and fn.endswith(".npz"):
                dst_p = os.path.join(OUT_DIR, fn)
                if not os.path.isfile(dst_p):
                    shutil.copy2(os.path.join(root, fn), dst_p)
                    print(f"Imported Stage-1 cached embedding: {fn}")

_free_gb = shutil.disk_usage(OUT_DIR).free / (1024**3)
print(f"\n>>> ACTIVE GPU: {GPU_NAME} ({GPU_VRAM_GB:.1f} GB VRAM) | CPUs={os.cpu_count()} | Tier={'T4-Fast' if FAST_T4_TIER else ('A100-HighRes' if HIGH_RES_A100 else 'L4-6View')} ({N_VIEWS} views) | Stage2={DO_STAGE2}")
if FAST_T4_TIER and DEVICE.type == "cuda":
    print("[T4 Fast-Tier Auto-Enabled] 6 Large-tier diverse backbones x 4 views (~90-110m on 1x Colab T4 | ~50-60m on Kaggle T4x2). Every backbone caches to Drive immediately!")
print(
    f"OUT_DIR={OUT_DIR} (free={_free_gb:.2f} GB) | VRAM={GPU_VRAM_GB:.1f}GB (bs_mult={EXTRACT_BS_MULT}x) "
    f"| Stage1={RUN_STAGE1_FOUNDATION_PROBES} | Stage2={DO_STAGE2} ({STAGE2_MODEL_KEYS})"
)


In [ ]:
# Cell 2: Smart Dataset Resolver (/tmp/t4_data on Kaggle, Nested/Subfolder/Zip Probes, Official GDrive Fallback & Verified SGKF Folds)
import base64, glob, hashlib, importlib, os, shutil, subprocess, sys, zipfile, zlib
import numpy as np
import pandas as pd

if "IS_KAGGLE" not in globals():
    IS_KAGGLE = ("KAGGLE_KERNEL_RUN_TYPE" in os.environ) or (os.path.isdir("/kaggle/working") and not os.path.isdir("/content"))
if "IS_COLAB" not in globals():
    IS_COLAB = (not IS_KAGGLE) and (("COLAB_RELEASE_TAG" in os.environ) or ("COLAB_GPU" in os.environ) or os.path.isdir("/content"))

DATA_DIR = "/tmp/t4_data" if IS_KAGGLE else ("/content/t4_data" if IS_COLAB else os.path.abspath("./t4_data"))

EXPECTED_TAR_BYTES = 768022016
EXPECTED_TAR_SHA256 = "d7316deeb95c78ff230229c84f5a2ffa0cd86874a2753405ac9917ccc68a3a44"
EXPECTED_NUM_PARTS = 37
EXPECTED_STD_PART_BYTES = 20971520
EXPECTED_LAST_PART_BYTES = 13047296

OFFICIAL_GDRIVE_FILE_ID = "1C8csylbBo3Oab24ODGKPVom3_SDUJ_Qw"
AUTO_GDOWN_OFFICIAL = True

KAGGLE_DIRECT_COMP_DIRS = (
    "/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/04_image_classification",
    "/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/04_image_classification/data",
    "/kaggle/input/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/04_image_classification",
    "/kaggle/input/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/04_image_classification/data",
    "/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/tasks/04_image_classification/data",
    "/kaggle/input/player-intelligence-system-cpe-342-ml/tasks/04_image_classification/data",
)

# Exact 5-fold SGKF split array (31,546 rows) packed in base-5 + zlib + base64; matches cnxv3/dinov3l OOF row-for-row
EXPECTED_FOLDS_N = 31546
EXPECTED_FOLDS_SHA256 = "e392c7d495259299"
EMBEDDED_FOLDS_B64 = (
    "eNoNl/tD8WwYxx+KkeZ8yuhdYXOYQwkP6mFjc24aCpXNYZhpciqqP/7t5/un676u7+FzNJ8sLdDI4e1rRP0YZhaQqatDPF22h+5ZWCXlfNyV+THPawqHOoVtAfkQmLDh7nmHPFu7ChvgIbmrqOxk3HG6+qCV2uWTHanuAX04a5ycWRGVsCMhzDKayluV+gVh3Qv5y9jqUqu7X06au+XtYlZFY4FKS4Of92Nvdakd63JnnZX1v7XRBKBKGiElcWEuh/9VD2Bg6lgLZKw5qLWmpZrLjRa+/uplkY5Fy8qxdLEmdDKxT8SudfS9vPEYerD2j5f3vuziFLfZEU+obH5kiTRHehdm60mz2K2x6SJN3t4djOsj+EMWOIDlIw7MXOzhGw6gPzV4xN0yU3rJ3SuCLv7vgE0WIr03WzEEhldxvoeQSk9nA0SpDX/3Wa/d+NkvLFi2qnUo8kKdifEfMOp2irolHjoGCiWYZo5yVzLWdwDqQ8hLDs+/r/+Tgk+8SAMt68vYUpQGy1jH1dkIsbiNKWpaYNQkLm8+3MMjeHHDtqhizDnf39zvFc2i7t9O+Oe7DOU3I8fddUExN9UpVcbd7M9i/kKgpDRAkdG9gsvUQNGwBwiRacyA0XneubUUc/RKkI+Sse9HERzMEFrdAmUvmPKrMHN9XeRJraJ+uigFNqYrKfVYpVJkzG8ModfNkVb5NDLD5OsYFmlRg+vRcXqIaeLHaTWb/pI9F2PDMUMpyZPgVPu3WOdBUiYFbZpiPXIJjp58J//ON77vVgwpNSx/yeP5KXtvaqHXMDkCgfbCtDRdnL5sljw4EBUX+NTDl1XI5quhIUknW0peVDSd09QUjc+qOjhJfoagl4FTFZq4h8pKj7vhL/FNpAPMnCFUWTdWUj1U2XFpMFKvQfn8Quqf9SLH2jl4c99yU/OAamVGZkDvlus+6Pax9KfOsSwGerwyy42JGUPF4VHwYy8xpwtolhd6IcSkQUKLHY9MPJtQV6VVluPWFRPR+pYAFmD3RNyz0YZv+oRNRm6QvQdEC2ZolOUcF7pD/L1WBmSpw1Erb23J/ZF9opGmkbjdl5hruvXz7evhtHWNOToZfaY1+Za5A30VWTw1rJk3XVpA1ooMTbTe9fqckw+Ik/hEZPtpAXRHxeXRso2Mw/DIQMoI5FNGweQWeDnUToao4uWFn/ABKfCNmINUc3JIxe8t4iCgmJ1n8mReji7c9EqJLnvaHuVX8YUJQaSnizFyKfeaVBUXvCglFiIdahv8YGlQc/lv2SIWX/s02HV2g8mRSbR42uSrTUC66xxoC8BnR/CUvozPHDcFuy4VppDRKQth0bhyxDij/hR0uOVZ11SJi+3PC8t7j1+6W/jD5OtRY9WVtyetSaQZk/XLQHsMl4/+MAsKmaWAj7JeOZ4Qk09DsTm/VX0mS6QKFGtv1y/oMcClC461/GPbNW5D5B7wZWPhGEt6DFd68eXokGLdWA6xAzUhSM4NF7cjQYM7STanSIfS5IVW8zKqAWWL2duk/AbKkj38NZUFGoLBrp4NsAgXnzo9I1nk+oFqIGSBz7GXjMO0ooFhkjz2df1Fg+ZL3vbzRYNW1fXPtfgx8rA6ZRsWkSmWq49qNbLHw7sjJh+YaY0MDSx870wLFB10e+IZVK/+eMofFmilHK4CID+ggaqnhSdZnpy7R867TD8IykoXUld6qM/BIXC4vm3dTcs/wMX4hU/Ft2ovaw/cs4F5tvq2Sk9irJ8yaT3soHh/mUBMskdMmDx0V3V79aYZQts7/8rKd44L3ItFs5y1pwi2AtTCjWFq4TX0387WmRda/ZJPle3LJOBbjQOsvmJ002H5I/8ykg28nHsNvQrOUuEHQJQ432Ep9wY7QpHN1e0rrHs7Sz2X5kGzmFXryVf38fa9BxzOPJ1VbeKk0od6+VLGHq6Rd7qprSU3JTOMb0nQ9ThNzjxDnbdeduRDQ0AwzbPDseumgQjGNIFI2w2fc6zYACEwxy5U29R0XRq9KTvVl7BLMKKiThRZ/ernIVcFTJm8O+rmubYjkN885EwTdSqw0nFfKlI4QfrX0TgfwfWiriUQofQdXZOAEaNK8DdXHUeZT0RrkXYvbP0k3SSkfB4X9l7QrrkggYA01/R1DyvrZb107X87KbBYjEU+HFo9uD4rXMVEZiqs9l5E0e2NzjxbstqxpPrAd9eXCP2XXVVJ/k8nlHr+qcZJuPuckFX0Bg/p5iaTLUvyL2Y4O/47AhvOgYqtsMpscx2qqcnaX7DcBc0qNP/WVJTdtJIomWxtQYvsZRyAM1+BhQG0GiOHmL+rg4jxDloU41t5c3lnviR+oF5kc5lgwPEB93ZDL5a1qiz+UZaLzpMmtWhUIFW3CRSPVh+hkszdB9hDwpmja1cpZ53/MZ08U01S9kgM9r6CBYR1q+s32ESZvH5p00b1XSDGUWAUIkYaLdEaxRp5UNT8vFya9OWTy8Kk5qUbe29vjnfV3YDO5FilqcklURiNURZw91cqHsWsJKDqA6ll/NsMCzf5QahWZA+gJ2A0z4u3bFTz0rSejwwM88Xq0QAoQCOPaVl9b/mgYkc0x/pzxMwjKv7r5oxGus6+Wl1ytFkP8fV9BHDxA3ai1RNfhGp1HWlO7yExZbMmimfI6rybWJQ1wRs+CvSiAJdIt8zl55jZM2FMms+iuyDBvc0beCA1Mi0NYYjWTUivTrsClYIgHtV7wTeBlBX6qmKycP03uz/wL6w+NdS8volAtf4Jtbw6wLbxBgRpCGCS5Vcphd3V9jY9cMmIUP3lS69EjfxXRT+TB0ZHMQ2f+PaJhHnFGDhEL2qZLeiqRcqyaRKtt645saTiJFNth+6hF3G85fDm1Guvee+HgqetXuIsXQ6cSU0ebCWUOa5rDraAildv7fGLZNfRWVeRiTV4KZOmNSQ1EgB9bmR53OLi7EL72pZlhRexW9Pq59O7/sWP5gOQ3KwGWcH39HZJZ8cKd9eCh5IteYEjEgqSFfG0ugxaGtnNIj0zcIp8F1iMgGrq+Y6nbUxNLB+fe/OS/ykz46xYD4ltwr9S7FqJ49qAg0yetjkwmGofWlqgwIaSxMfVSfrTcavK6SUFLtb95LHdY/XqO2YswyIrQEPqGqEIMnGK1fZOHCmysqvUwlRqlIxCoD1HtPjA4mUdRGFWkYFuska/xb1RWuHEcnQ459O/Gzm7l0cgysu+sV6TPisUzDAVpkmZt7eP2NSTpvQBEPo3MCj9l/06yvnssru+aK0/YKFQNgM5P11wGlj30KkyS3+4j0YqV4HP75Bfq8/uefLNHx9oiz5zEW9KsAsovJ1w3g1fmMEiWKldk89n3HUIybRtJpTmva1bJjQd+k3nuZqbK5n0RGbwR+xaWshRa5Lhb4LRd5l1+7XawAkauY8+apm6hc06pNGNQsPdGZtUZBLUXnxBbFEFrhGn8wpQIQddCRhMocIBpeWIpRPK2pPxMAxu61duN+kdySAj/4bHCmHT6SVImiglLqSj70feT8gYKvsmXV84YYvUyNs9BqoxJG0bE5edeflPNeIHJ0su9GRQLQ6WWC37yVjy0ljEGMoH3E1pYAt5OZvxQ+ZdHN6PhW8Q1djOtsiySp4FclwbZjWvP49/NnDe5H2RH4LygWVwrkqxATtQnzPmSEHif8ce/snXisA4yBfL5liWe2qg07I6Cc6aCPmGT3jVOgM2KV4+goDgpCfkowK7apxEBrIN9tbOQbs1Z0ZoX+W/E/ZDWUz1Dic/iDS/qYWzGcXoifWMSDmnufhV3RiJ4IvfPitYDMSMi/H5l7H7SpULWOl2UCgHavtIVVV+i/UypCO2+oZ0XHy+kC5OCMNa8h8P0uMIwoNm+HV47UXt+upGQ2Tcn3yWb91Om15poNTWFlOiGN9g6EoupRi+PDqKWAoPk+ejUUEe8nLBcSaGCOQRUR6O0UmQz12emmR9lTvDmsyAOeXft26/a152hakqAQX8eaxZlFCOFsAr8EMZsFpwrOyQChWVSQOuLOK8MC5XU4dzcoPD70BfVlr2Q6evooG64N3cXo5w4LeLQsxwm40GcCHHdyE6uAaAzlauXldhoI9/zw3ldNzsGQH+n7wZsgyI6gxxaRFx04GuYHoEsWPYZPBu73oVziQCO33IPZCC9jRq+wR0IvEG6GLCLreJmdw8NGowIqMdjsUFmFDm5+qCNTDxiTQBVVp/gsXRqRIin99rkT1Oe4hV07rVoSt1WiCsxUDNIDpZR3xhL0rbySY8eamV1Bxaj/P6WikHgFzC8cgWkXFwDXHT5CBm+74wdF0jXcaMIyjYgKn71sEG1BkTMEBcnHMCVWuar507zoXqAEcq8B8N0J2GcaYW4W2D4uIM6PD6AdB9VzSlt5tG2k7YtL/o1IdMqnSkqcPZoqeOHiMf/I3dswW47tjPDmO71vnNy9ZnXQSFkmbaii0oYAEZ+MIeS3qlLpe41wcWdQkGMS3PGA+JcFI/gJ4FPDLNTOMTf1aNUXk9d92vpSgX0Of6K8XN/LTvnGpQ9xYtO7Q9tC/zUJwreyjwFg05lIHnL59y0YK3F05+pehIrZiqEHqkRB4j3Bahs/fmlS8mn6UCtU4+3ff08yJ+O7d854BXYCnn+pqIulFBmieeKVpifRwK1vCvew/f1/IIn8V4Mo78nKV3sZXInHBnhe+jRVKD1VOfOuQbMiHuIlff+9BW0oCU9dwVGNhVgW8hrwqUxwEOWMgf1cOMvPcWVwD3wjv43rgyOC3G4gc0Lxl8rP96m8lcB61+S7yYofmEqdmpqxP+rou0kLW0oc4HkUjZcvQXElaOp0+eRaiIUbjrn+2hOAchQDksbxa12qfvI/hwet+1W2prFaZ530rjFRdvZy+s+p8qPCJOwtvT0YWmMfm2JPLbU/TBHNinwSvMLWyA1d4HohHXPWHJ1VH94joLGL/VCXkUnoJEelxhTPzfmKmQCJbkkTt6AX3zWpVU9yzqa68esC2N1bF74uRKDX1kw6eJGdD6p33Rk2qgA0wC5LfsGKbdAzwUAX2bskDkvB0zDwyeXYkFMlLKiV0j8V8Y7Ixkd91jR5b9Qox2bxxNjx8YthLU4q7w9VONlM3IwPgxzX6EpIsR86ui6jfQGAHEvqNG76cAMr99vXKzWP4Y1XCnDWnq32balJooTslaxlFJkrIhosLn7+1fi/lPDuA/QYDV8ZldR+b9UoU8eVZNBhIOBGWV5VxV8wa9RsEQTG0nGIsMrv/zOn8spnys33Rs3GLe7EO6GWBna4ciqg2auc+Y9Dmw8VvWDX6OL7TeZRFDX91V7Nx8V68unKDi0SoE9N2dqzFRBPt0z+0pzrek0kx0p/yXG1HtUQRXcSTo3kTeQPCyXK/ETOrbEQIU9jBzFAJ5cGf9L1TgSYf09WB9EpzYMwlRe3+JVa8EPLBPLarJTlSa3zZQo/CdNyMIywPlfYM0p4I73VsMZgNUbXvLD4VsNFaq0VGNmCuzKho41P7Uy8DDZMA60d/q1dd0lV96Uid22luLaMT5i0y7dHnPrnk5V4+g4DpIq/cgxCFzAlSqMl8cSDm9nyNgzAHYLyP+ggJ/XYgmvSNAwuppnHJujfLvuD8BrZv65bjku0BZt6G41uuQrXtXWzlDaemsBebenI1ybjZtWQO1PhY7jX+q8ceWLB0R6325JW07J7N6FoUtmJ5UPDPShAWeLd4h33Wpkd9ND+5oS7DNot3zStvJab33IyCpugpU0cw1Fkeey1fGe6/4an6DoaGCwAbHyX6E0hhQR3aEOzFlxQ2qY9IRhq8zhM5SjkwDxtryXfWtIP1asc7K5NztPggZOaB/a7kozWQg6+YOZyLx9WkpVCfdxRGZWIaTg6OB9yZcxQfRGSIjhdj2usOi/uv+hOEZFiCozl4fqQniFqQ7x4qshF16HwkXLg7OTUlfRlhJ08gFMZCHpdMBEGRTFtdjgnr2bumYUGq66ZpmAqxYsvaLQ3bPhum8TaoHwKQixXK9ofGvupa26S/OHvIfsqx4KZrqZ6gq9anItcBfU0bqu+OLY7ErK7CoApH6TgpuHVcaTK4ajDET8FuT6qOV6tXxZRzZ9bxjz8c/RNFXrVtsRYbWylAfwhKB2tfG/Sx+K7oxc+EAlDoq+8VwRceKPW2N8gyUZTY0z7UnT31lgs1QfpJWdlP16qcrYLk9DbrAvpJ0xN1UKC2Qi8RUcBaBAi9WjAcQV/DgJLhJ/d79xvNxQqyPLmNDMMxn7XrEEmOz/3EakmW+3SsbQgEmG8p8ngQTQ/V2NYbjYvRJrUj6S9OjT1gPisK47sg9T0H1Q/7AT4gOLUN+/504XqcTys4mYhaDnnhkH2iBKxfFbBDp+PdQWa+lE9taNaZhqO5pC6VobVLgg6n+HjaPHDgRG0rOMcwiazqgwRz5ZXLwAwUn4zJiDjmzv+SZ0Lfnz0sH3kdmyrJrKkWS/EXFmKdE3zvz67qcSZ+S/rJRfCMwZSVpxZCbo4kMcw4uIVusxPQsBwnHZNkVT9dSJX3xcJYdnsexeFqg9GhfQo69/hzLXu6cUF482jYTv4kjxeor/Shi8gwn3g+zIvIRYQE6IDxrvoz4x/z0+X1MAi0MXNZIHUAswNIbNX4+SNk0LrkulFk2EqELyc7+/Ka/BgKxoQrNUZWjRuyAk55ij3K+k6owX6BPqL8m1tPSiybB+6SKSlioYDsrWR1ZrlbSMuMHcMW/JJx8muIoTINjAa5n92iLJmbK52e+O/HVPeH2styzxCVoeOP6m2UvTslJUuzZ+XTooXc3FQpdDU+RwEiGDIzu8MtNIqvii5VxQ+KQO6uPuXyvfiKyyQuqHzE7dfrRhCnA3NJn5EvxcNYMPye2x7FWwX7nE3rq075Z7Fk0pptH8G1lRsjq5T5lwJtbNxkxzKrMZYmJVaqcY11LtTaRWnxOZf+8LmJkRnPdTwRYiJOt/uZO+8D2HztlUZ64EEUrcM/HP3Asqgn/KevQa46zXEc/nwDtZaizlHv7ez/A3/bEnvUyvmEuYcNb7DC0/vEX1okporYgL8eCSQwMZMj8VBJaBZcn/mN4dI7nJbJ5TImosYxzGp68e9YqBrzYUuDzrLZWmIOSkFxo3QWtjGym+6GJyXPaszTXt54jzPzIZHYJXUaf702REHocoIsj4LU49rIIMDiu/f23ueq3Z+5I1++8n/+L+hqV63D8vUZ81+lMZ3ZBeWgRZ49j6adxmp9AHGl/uP00zvN82chLrWwtzOPnJZOieUyY3QjJN85T+Y9wbNKXPfxc323JI4D2nnH3ObW7PUVHyHH4ZPd7W5lkmSd+2TlbNNHwQ73Ls/7TfuJyIG8FQOcU4VdB6Bt4H2OyLu8Z6LytCTkOQgJSTgA7VTj9ixpSEtN0GRU1mtRSUUm4eV1UmDGkys0KAlfWnyIcKWtz8chIRiFrxQCh+pCmv3GheERdqFGRznXL5T+x5KcAVt8jwUNMKxZifVkmy/+1DXyx43cunYuf6y2GwXSfjqPKF4HsKpafoB0q8PD25Pe477p003JGcQVwMsfkUQrtQAGhsAd2lgDIASwwAh4Z5ZBJZD8lFLSYUaYzVT7v7wLT7WQuT7agPIuyDLI/phF6AyfFgnnBvLWfrVPx9pyMX2zf28OzwzJ90ivBn+f3F7lFsafhO3ycRX15jFXHEG2sG6P5XxLZQsaIqmh64cYIflix6H3GHANaEkF2Vcf/jJf6fxzJ37CWsrOoSwtZOlp+15dlmpA7eTIwviXbfP/O1GafM+0ARtQnM7FV/1WtI+vmwMxgWnUjQzf6BprqzZIM+9ZQ0WSu9mKTLta0mwR+Hto+5Dei0baEuS865BheScIDq1meDMOnJqCZwBSkZoN2CoJukanQ7rRFbCorR6DLGsK5laxjP89Jda6hG3uJKSJMfmsDsteUW7KUjjvlZN2dA/yqLd0f8svsJkRMve+7qbIwN9BeWjlD7jayQCuM2CEi/tbj0ct/RgOFzQ2qAXdGwWDUTJAHNWHVR2xtVMKm7iIYvJ1aOodrmP3lydOPnEXlqRjNSjLSBgPTjXJpRdgnHhJ39kDr9Lc2iA2y7/6M4UKsCwx7RU2y0rMSyrCVwNmu17aVgt9t7Niz+vQV7RtNlMzg8mbYUKY1HJsejLsqbx8APsYheMAPgMJMWXjX8NmRyjPZUDbkahxbtGrFv2aGsyUoj6q9U/tKI+REWWJWsmnpeblJafrQL4JIl96AiZueYNUO9R/fxdE1wMXfl9OmJbW/Pme91U+7MpC57/BmuLDXLDXlyXNkQjuz/HKgwiMxOtQ53sAJ0AbCqIpHJkGoNVsLAefaCM3fEZOPcnTrh/Q1zy4J5MfpBI55coysHCQOR06GGuKLZfy0fh3jTu1BgLSl6RjxkzLyShKxly91wesFChYnv/sdJlXT33HtozNPx65avnxY1Q9el+96SV88Hl38nARTn6JrYKEhYwgUrUFAQjZz3b/FHjm6+0WKxSnwzdvgF9ZinedtsCnBMy3rKzDOUOgzYl5uHn/JGrnIDWbySOADqr89m1T5wtmeeK8ZsDM2som6iS+dxdtXFN6SlpkXjGOtO5BOoJ69ke+sr6PBdGUMMOMg1TJpUr2tl+ahVeaJds8QM7KxefcLdbYWuWYD1qQ3H1WV+9dR7TlDaV9FO3CvZmEIr24cg5WSGGnVCiLLfLzJHusEC/N5uqwvmpXsQBuI3AO7WYzD1YS0sODwlBlexU63oKuBrAH3K185K4e87tHwj5f1DDTDSYxYKbSTBkrldxBge6BLZ+wv/TeoejrE1D4Od5vbzxqitMQvV7bf53eTX96KF10tgPujNQ/Vj9rOX75ccJmswDQZSjEGS3780IW+IS5tBlZzr25AmJGVjpk49z0lEz0ArJcTpuUxTKq6QtfN9U4iBuyYSNlDhbtq+huHzfppqjnMuQw6y3ElsIh/KpCDc9cWMnGgEJkS6tvnuTXO1cWpD17Fi2GtwaYG+cnDpGvA3mWEW7dTv2bHDUum6cFzdtmv/e+m53muPHYTU/k3vx3cau6ak5YdUaYwS70RKM0c4NObHlXg6JCyP86vCdJO6+8SyH2aj9Mlp1oT+JYBXXWkvzG9YBGVtRmO52SCpxMyTRIwMNT/LXwp0pyy82YZad34dKbE0zOIu3n+PCIEEq7XMoy5Xz8vCNqIyTlZocrbDjaeXeTHNhhZJ3vai9uKv+pfiSAf/4F0yOWEBPoKgplGDKZ1VdRg182OGA+ftpKcdrRSFHbQ3+kkavotKukP97hFTKCAFH39UgBr9+K++7r8g6qfsE2YVGS1XNEcYazBpGVUlzEDpDVGe57RmUDeGkMowyQ378pX6iL76f6ZwHXmpOFY/PsBCuemnGprOP8AAnskHt6Z9XmSdpCHhOEzeosF+n8EL0n+eiIOtbR9p/Y3gTl+747jGmL7NOYj/WsGinbLCk2/Ms9Dja0t+VTmMTPDylJp/0xkPkz6W9LxxIGhQFkg57ICsTQKRYArk4r8k2oT+lsWdO8fkuWsYGfO6iJJCyk6uE6SJyMHagBU/Eqz2Ckkxh96zg2hIs2dBl+tgHSVnyFAT1tsL4xv8UjuHhk/7DXp3dxZ8PW/SzI5fQunR+xY6b5diTv9XOoJGNlGkjzXZt2XkZfq5YKUN8VP2e0A6sG7PY2o630Rj8jgcihO3G5+gPvIIIVH3bo3OXuBrqEnPafTzDXBiTNMASNf5qavELuqWFmzUztJWhFb/rKO6Nqpo2Z8kqb4FtQFAlPMzVrImdI7uhm6WVX2yjk9CZvya6UOxXdA6v3DnjrD7ujFHhSDZiCCmj6Vsjxby5G8o7CtKE/aWKwMf9pr1ABAcD6wkzASMj9zVHwsgsjMFL7x3+Ux1MhTuMy/HdyFcc035CSfzyaDB2NOG5AxNPQolm+CJmBIIVvVKcW/rEIwxfCd4kBfN1/kd+pj0oUku0rOMLfWV8JcziILqSp2A7rF0oZnRkaPRPBgCEEL+mMPi8ynuePAbyRPasMwXmLp/95B03+TvNrUc9gySdxEQuh1ds8HuFLxiLlC6pYWWgcYBweawKsb2NYQSB0Kb6CCtUZhIQPwyVEsw1OsP73B58BKIbbK1pGeYmrltPYgezJBrP6qA5SNLsbe0Py6NPSzVmb5nfas4z899AHLaqQ+6UOfMi83Tt3nau0QHBdevvgQpsrGJmd+CRv4SB9OfFXDoIS8y1oZQjak3hNYsoExIL2AzKfzm6ES5PMfKhLgVmPE0hmjxOXFzRSx8ziXeg1df9lhHBkII7FmAbro/djl5VTKUsD0Zf4EfIENYklt8W+LlzTDtAAXrkDNlaYzkgvbknHefgPAZbVziRo2bnE4hgu13C/77xFlVFMaHskxV3yBtKFZvyDSWMw3eZ5FDkYcjcES9/gD6uCuy1BUifeTywyuLtj7oVRvicrSse8OVY26L1cVBWGY+VuHlJzdwC/NWZx2ZPvmi7v3P33e29mo96lIPWpz94qRE07qT9z7pisdG+lbxX/8NAQRpVcbknDJ8i35FdPDUK1Qxvn2JNVyf1OtgcPIohnFk39tc+S3kxuOjSg6HbQlGyz2d0lTLSX4j6zpXdkp6FEKet57SG1Ab2VmJjfCPNDeTGND3dnUyei/Kqv8t1jpDW+rcRjcq5GTQAjv8sR2XKx2iT+LDFWj4Frdkt0jMKe724VM6XHkIlLVfLa0niNPx0wIjysesfbU0Wtioa3ViiCFVr39mR5JhUj9ijWX/fw76+9IvCqjfBIGZGz0eLSV81C3zUcNzb+23fgy90zHz7hbyyZyLyLHwikxeR6L/h0y/uPtT3yHi3T2588x5kZV9fkAWIW0fh5POB6l7W3nRIykLfUHajVUdomjFU8hIZWUe5iOAQ7Omucwl3wxX6Dkn1HjVpz5HvHnoVD+OvKkBj3gK9aqlnlEV1LNtZ1+Mqfwf/Utcg5PQCNiCfneDYLntK/7baIe+sm3B0gsvY3Lun2x5XtXFOHeeutIklW0RlBMa4KMy2INr60VAZJ3Dwa1hA/bpPi7PsQ90bJc0tXRC6Q1bXkPqM20MCE1JAuXhUYFrGWvtLT6Wp/QE9YtH8rTacspGfuPWYCD7+frkUd76yj8WICBr350GyX1S8oRKN9bUubWKdDnY00jT5TmEJ1uby5BMIo+bS7Etisw9UwatGNS3kB9jWi8cp3Iun8ERPKTdS61AbBlPcD9CYkdwJxbJO7f3Nv06VtDSPXeYoUZ0HcHfxPDCpDM3r15yGkKZdnnfxUHzuV6lvoVhI96yFmo0nQLJFAQ4yYWJm3As6inApsTkap9FpnvMeXIbmLYpOsYEVBtc6DuXkgZ7Z11GT7LfOrQ8cWSNHu4Df63LG3ge1HIJFNgv5EbhGjPYHOlIFG9xvO47x3qb7LUy6W2cOKP2mArjkRswGICEnZASHbE9hLT9mQakQO1559eo9SKh6VU57dIc/NEvAUlO281LmuCnJUeX3C+aUvBGf8XvD8A6bhNJkppa7UalZHKBvkYp4jQyubwTIYfJbMrhNbyrLf3ICEkxWoyqI4fII5JrO86X6avKroeJEJRKkjLg3u4Yx7C+sqtb0sxUVCg1r6VImsLIeHjm9GTqF8UZTkVh4PZT6jLMxtxaQrObmbXaF+droNCpOf9oJN+sWxHZG8gXDE4JX+KVaXR4zDyzlrUp13fcHawEEDT+X6MFrJRNyvWiJuZc6XqnpcPZ0TBDi3v1Rp8M/Lst7kv2fOYVHmFwgba86nC21mTz8X6sS9GF+TXAKnEf/TIpHlIpAxbOT2PT+PofTinTqjiorJF4sVk0BTptZkZfQikbA1peQ9Qp02sUWlktydYE1zHLYiVvlQSK/mKZ0Y8kVB1Q8G6FrDp3OPz0dPmigHFo5DNi3xDoP3sLZRlUd6YNB2X9IGrdu2VT2gRuIwxP4rkPcKT2hAJYTUy7h3I8vOybiLkopb4eYeV4Oi7UAPaIrfWEWVgm8ho2GbCPTzJKIiFGqjcb9Ce6NEM4dvpmzrPbbQOd+tk2I0Zi7MT5LKkhzZwmd1ey6SyQzW4j5hbpRjm1ELs44teinJ1/uZkhITNGdLII6vGskHVUj+qdqV65fjXNEwE7+dJc/w/DNfm2g=="
)

COMP_CLASS_MAP = {"7-11k": 0, "7-11K": 0, "fiveN": 1, "freefried": 2, "pube": 3, "roblock": 4}
_SKIP_WALK_DIRS = ("train", "test", "Train", "Test", "train_images", "test_images", "drive", "sample_data", ".config", ".cache", "__pycache__", ".git")
_IMG_EXTS = (".jpg", ".jpeg", ".png")

def decode_embedded_folds(expected_n: int = EXPECTED_FOLDS_N) -> np.ndarray:
    """Decode verified 31,546-row 5-fold SGKF assignment array."""
    raw = np.frombuffer(zlib.decompress(base64.b64decode(EMBEDDED_FOLDS_B64)), dtype=np.uint8)
    d0 = raw // 25
    d1 = (raw % 25) // 5
    d2 = raw % 5
    folds = np.empty(len(raw) * 3, dtype=np.int8)
    folds[0::3], folds[1::3], folds[2::3] = d0, d1, d2
    folds = folds[:expected_n].copy()
    assert len(folds) == expected_n
    digest = hashlib.sha256(folds.astype(np.uint8).tobytes()).hexdigest()[:16]
    assert digest == EXPECTED_FOLDS_SHA256, f"Embedded folds checksum mismatch: {digest} != {EXPECTED_FOLDS_SHA256}"
    return folds

EXPECTED_TEST_N = 25889
EXPECTED_FIRST_TEST_FN = 'sjqyfwc.jpg'
EMBEDDED_TRAIN_CSV_B64 = (
    '...'
)
EMBEDDED_TEST_CSV_B64 = (
    '...'
)

def decode_embedded_train_csv() -> pd.DataFrame:
    """Decode canonical Nov-12 Task 4 train.csv (31,546 rows, PLY00001=su7tkav.jpg)."""
    import io as _io
    raw = zlib.decompress(base64.b64decode(EMBEDDED_TRAIN_CSV_B64))
    df = pd.read_csv(_io.BytesIO(raw))
    assert len(df) == EXPECTED_FOLDS_N
    return df

def decode_embedded_test_csv() -> pd.DataFrame:
    """Decode canonical Nov-12 Task 4 test.csv (25,889 rows, ANS00001=sjqyfwc.jpg)."""
    import io as _io
    raw = zlib.decompress(base64.b64decode(EMBEDDED_TEST_CSV_B64))
    df = pd.read_csv(_io.BytesIO(raw))
    assert len(df) == EXPECTED_TEST_N and str(df['file_name'].iloc[0]) == EXPECTED_FIRST_TEST_FN
    return df

def count_real_files(folder: str) -> int:
    """Count non-hidden entries in a directory (handles nested class folders via fallback if needed)."""
    if not os.path.isdir(folder):
        return 0
    return sum(1 for f in os.listdir(folder) if not f.startswith("."))

def _normalize_filenames(col: pd.Series) -> pd.Series:
    s = col.astype(str).str.strip()
    return s.where(s.str.lower().str.endswith(_IMG_EXTS), s + ".jpg")

def _get_file_series(df: pd.DataFrame) -> pd.Series:
    for col in ("file_name", "filename", "image", "img", "path", "id"):
        if col in df.columns:
            return _normalize_filenames(df[col])
    return _normalize_filenames(df.iloc[:, 0])

def _map_labels(df: pd.DataFrame):
    if "label" in df.columns and df["label"].notna().all():
        return df["label"].astype(int)
    if "class" in df.columns and df["class"].notna().all():
        if pd.api.types.is_numeric_dtype(df["class"]):
            return df["class"].astype(int)
        cls_clean = df["class"].astype(str).str.strip()
        mapped = cls_clean.map(COMP_CLASS_MAP)
        if mapped.isna().any():
            low_map = {k.lower(): v for k, v in COMP_CLASS_MAP.items()}
            mapped = mapped.where(~mapped.isna(), cls_clean.str.lower().map(low_map))
        if mapped.isna().any():
            mapped = mapped.where(~mapped.isna(), pd.to_numeric(cls_clean, errors="coerce"))
        return mapped.astype(int)
    return None

def is_data_ready(d: str) -> bool:
    """Fast O(1) readiness check against manifest length, folds_sgkf.npy, and boundary images."""
    f_p = os.path.join(d, "folds_sgkf.npy")
    tr_p = os.path.join(d, "train.csv")
    te_p = os.path.join(d, "test.csv")
    if not (os.path.isfile(f_p) and os.path.isfile(tr_p) and os.path.isfile(te_p)):
        return False
    try:
        tr_df = pd.read_csv(tr_p)
        te_df = pd.read_csv(te_p)
        folds = np.load(f_p, mmap_mode="r")
        if len(tr_df) != EXPECTED_FOLDS_N or len(folds) != EXPECTED_FOLDS_N or len(te_df) == 0:
            return False
        tr_fns = _get_file_series(tr_df)
        te_fns = _get_file_series(te_df)
        for idx in (0, len(tr_fns) // 2, len(tr_fns) - 1):
            if not os.path.isfile(os.path.join(d, "train", tr_fns.iloc[idx])):
                return False
        for idx in (0, len(te_fns) // 2, len(te_fns) - 1):
            if not os.path.isfile(os.path.join(d, "test", te_fns.iloc[idx])):
                return False
        return True
    except Exception:
        return False

def sha256_file(path: str):
    h = hashlib.sha256()
    total = 0
    with open(path, "rb") as f:
        while True:
            buf = f.read(8 * 1024 * 1024)
            if not buf:
                break
            h.update(buf)
            total += len(buf)
    return total, h.hexdigest()

def _ensure_symlink_or_copy(src: str, dst: str) -> None:
    if os.path.islink(dst):
        if os.path.realpath(dst) == os.path.realpath(src) and os.path.isdir(dst):
            return
        os.unlink(dst)
    elif os.path.isdir(dst) and count_real_files(dst) > 0:
        return
    elif os.path.exists(dst):
        os.remove(dst)
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copytree(src, dst)

def _unwrap_single_subdir(folder: str) -> str:
    """Step into a single wrapper directory (e.g., train/train/*.jpg) when no files sit at top level."""
    cur = folder
    for _ in range(2):
        if not os.path.isdir(cur):
            break
        entries = [e for e in os.scandir(cur) if not e.name.startswith(".")]
        if len(entries) == 1 and entries[0].is_dir():
            cur = entries[0].path
        else:
            break
    return cur

def _detect_class_subfolder_map(tr_dir: str, tr_df: pd.DataFrame):
    """Detect class-subfolder layout train/<cls>/<fn> in O(1); return {label_int: folder_name} or None."""
    sub_dirs = [e.name for e in os.scandir(tr_dir) if e.is_dir() and not e.name.startswith(".")]
    if not (1 <= len(sub_dirs) <= 10):
        return None
    labels = _map_labels(tr_df)
    if labels is None:
        return None
    fns = _get_file_series(tr_df)
    cls_raw = tr_df["class"].astype(str).str.strip() if "class" in tr_df.columns else None

    lbl_to_dir = {}
    for lbl in sorted(labels.unique()):
        matches = np.where(labels.to_numpy() == lbl)[0]
        probe_positions = [int(matches[0]), int(matches[-1])]
        for pos in probe_positions:
            fn = fns.iloc[pos]
            cand_order = []
            if cls_raw is not None:
                cand_order.extend([cls_raw.iloc[pos], cls_raw.iloc[pos].lower()])
            cand_order.append(str(int(lbl)))
            cand_order.extend(sub_dirs)
            seen = set()
            for sd in cand_order:
                if sd in seen or sd not in sub_dirs:
                    continue
                seen.add(sd)
                if os.path.isfile(os.path.join(tr_dir, sd, fn)):
                    lbl_to_dir[int(lbl)] = sd
                    break
            if int(lbl) in lbl_to_dir:
                break
        if int(lbl) not in lbl_to_dir:
            return None
    return lbl_to_dir

def _probe_pair(tr_dir: str, te_dir: str, tr_df: pd.DataFrame, te_df: pd.DataFrame):
    """O(1) filename probe returning (eff_tr_dir, eff_te_dir, class_subfolders_bool) or None."""
    if not (os.path.isdir(tr_dir) and os.path.isdir(te_dir)):
        return None
    eff_tr = _unwrap_single_subdir(tr_dir)
    eff_te = _unwrap_single_subdir(te_dir)

    tr_fns = _get_file_series(tr_df)
    te_fns = _get_file_series(te_df)
    tr_idx = [0, min(1, len(tr_fns) - 1), len(tr_fns) - 1]
    te_idx = [0, min(1, len(te_fns) - 1), len(te_fns) - 1]

    if all(os.path.isfile(os.path.join(eff_tr, tr_fns.iloc[i])) for i in tr_idx) and all(
        os.path.isfile(os.path.join(eff_te, te_fns.iloc[i])) for i in te_idx
    ):
        return eff_tr, eff_te, False

    cls_map = _detect_class_subfolder_map(eff_tr, tr_df)
    if cls_map is not None and all(os.path.isfile(os.path.join(eff_te, te_fns.iloc[i])) for i in te_idx):
        return eff_tr, eff_te, True

    return None

def _resolve_comp_layout(base_dir: str, expected_n: int = EXPECTED_FOLDS_N):
    """Resolve (csv_dir, tr_dir, te_dir, class_subfolders) in O(1) across nested/flat/renamed layouts."""
    if not os.path.isdir(base_dir):
        return None

    csv_dir_cands = [base_dir, os.path.join(base_dir, "data")]
    try:
        for e in sorted(os.scandir(base_dir), key=lambda x: x.name):
            if e.is_dir() and e.name not in _SKIP_WALK_DIRS and not e.name.startswith("."):
                csv_dir_cands.append(e.path)
            if len(csv_dir_cands) >= 24:
                break
    except OSError:
        pass

    csv_dir = None
    seen_csv = set()
    for cd in csv_dir_cands:
        rp = os.path.realpath(cd)
        if rp in seen_csv:
            continue
        seen_csv.add(rp)
        if os.path.isfile(os.path.join(cd, "train.csv")) and os.path.isfile(os.path.join(cd, "test.csv")):
            csv_dir = cd
            break
    if csv_dir is None:
        return None

    tr_df = pd.read_csv(os.path.join(csv_dir, "train.csv"))
    te_df = pd.read_csv(os.path.join(csv_dir, "test.csv"))
    if len(tr_df) != expected_n or len(te_df) == 0:
        return None

    rel_pairs = [
        ("train", "test"),
        ("data/train", "data/test"),
        ("train/train", "test/test"),
        ("data/train/train", "data/test/test"),
        ("images/train", "images/test"),
        ("data/images/train", "data/images/test"),
        ("train_images", "test_images"),
        ("data/train_images", "data/test_images"),
        ("Train", "Test"),
        ("data/Train", "data/Test"),
    ]
    roots = [csv_dir] if os.path.realpath(csv_dir) == os.path.realpath(base_dir) else [csv_dir, base_dir]
    seen_pairs = set()
    for rb in roots:
        for r_tr, r_te in rel_pairs:
            p_tr = os.path.join(rb, r_tr)
            p_te = os.path.join(rb, r_te)
            key = (os.path.realpath(p_tr), os.path.realpath(p_te))
            if key in seen_pairs:
                continue
            seen_pairs.add(key)
            probed = _probe_pair(p_tr, p_te, tr_df, te_df)
            if probed is not None:
                eff_tr, eff_te, cls_sub = probed
                return csv_dir, eff_tr, eff_te, cls_sub
    return None

def _find_competition_data_dir(search_roots, direct_dirs=KAGGLE_DIRECT_COMP_DIRS, expected_n: int = EXPECTED_FOLDS_N):
    """Locate (csv_dir, tr_dir, te_dir, class_subfolders) via O(1) direct dirs first, then pruned os.walk."""
    for dpath in direct_dirs:
        layout = _resolve_comp_layout(dpath, expected_n=expected_n)
        if layout is not None:
            return layout

    for root_base in search_roots:
        if not os.path.isdir(root_base):
            continue
        base_depth = root_base.rstrip(os.sep).count(os.sep)
        for cur, dirs, files in os.walk(root_base, followlinks=True):
            dirs[:] = [d for d in dirs if d not in _SKIP_WALK_DIRS]
            if cur.rstrip(os.sep).count(os.sep) - base_depth > 6:
                dirs[:] = []
                continue
            if "train.csv" in files and "test.csv" in files:
                layout = _resolve_comp_layout(cur, expected_n=expected_n)
                if layout is not None:
                    return layout
    return None

def prepare_competition_dir(
    csv_dir: str,
    data_dir: str,
    tr_dir: str = None,
    te_dir: str = None,
    class_subfolders: bool = False,
) -> None:
    """Normalize Kaggle/official layout (0 GB image copy via symlinks + verified SGKF folds)."""
    eff_tr = os.path.join(csv_dir, "train") if tr_dir is None else tr_dir
    eff_te = os.path.join(csv_dir, "test") if te_dir is None else te_dir
    os.makedirs(data_dir, exist_ok=True)
    _ensure_symlink_or_copy(eff_tr, os.path.join(data_dir, "train"))
    _ensure_symlink_or_copy(eff_te, os.path.join(data_dir, "test"))

    tr_df = pd.read_csv(os.path.join(csv_dir, "train.csv"))
    te_df = pd.read_csv(os.path.join(csv_dir, "test.csv"))
    assert len(tr_df) == EXPECTED_FOLDS_N, f"Unexpected train row count: {len(tr_df)}"

    labels = _map_labels(tr_df)
    if labels is not None:
        tr_df["label"] = labels

    tr_fns = _get_file_series(tr_df)
    if class_subfolders:
        lbl_to_dir = _detect_class_subfolder_map(os.path.join(data_dir, "train"), tr_df)
        assert lbl_to_dir is not None, f"Failed to map class subfolders in {eff_tr}"
        tr_fns = pd.Series(
            [f"{lbl_to_dir[int(l)]}/{fn}" for l, fn in zip(tr_df["label"].to_numpy(), tr_fns.tolist())],
            index=tr_df.index,
        )
    tr_df["file_name"] = tr_fns
    te_df["file_name"] = _get_file_series(te_df)
    if len(te_df) != EXPECTED_TEST_N or str(te_df["file_name"].iloc[0]) != EXPECTED_FIRST_TEST_FN:
        te_df = decode_embedded_test_csv()
        print(f"[Canonical Fix] Restored canonical Nov-12 test.csv order ({te_df['id'].iloc[0]}={te_df['file_name'].iloc[0]})")

    for fn in (tr_df["file_name"].iloc[0], tr_df["file_name"].iloc[-1]):
        assert os.path.isfile(os.path.join(data_dir, "train", fn)), f"Missing train image: {fn}"
    for fn in (te_df["file_name"].iloc[0], te_df["file_name"].iloc[-1]):
        assert os.path.isfile(os.path.join(data_dir, "test", fn)), f"Missing test image: {fn}"

    tr_df.to_csv(os.path.join(data_dir, "train.csv"), index=False)
    te_df.to_csv(os.path.join(data_dir, "test.csv"), index=False)

    fold_cands = [
        os.path.join(csv_dir, "folds_sgkf.npy"),
        os.path.join(os.path.dirname(eff_tr), "folds_sgkf.npy"),
    ]
    src_folds = next((fp for fp in fold_cands if os.path.isfile(fp)), None)
    dst_folds = os.path.join(data_dir, "folds_sgkf.npy")
    if src_folds is not None:
        shutil.copy2(src_folds, dst_folds)
    else:
        first_raw = str(tr_df["id"].iloc[0] if "id" in tr_df.columns else tr_df["file_name"].iloc[0]).strip()
        assert first_raw in ("su7tkav", "su7tkav.jpg", "PLY00001"), (
            f"Unexpected train.csv first id={first_raw} for embedded folds_sgkf"
        )
        np.save(dst_folds, decode_embedded_folds(len(tr_df)))

def _collect_dir_snapshot(roots, limit: int = 20):
    lines = []
    seen = set()
    for p in roots:
        if not p or not os.path.isdir(p):
            continue
        rp = os.path.realpath(p)
        if rp in seen:
            continue
        seen.add(rp)
        try:
            ents = sorted(e.name + ("/" if e.is_dir() else "") for e in os.scandir(p) if not e.name.startswith("."))
            lines.append(f"  {p} -> {ents[:limit]}" + (f" (+{len(ents)-limit} more)" if len(ents) > limit else ""))
        except OSError as exc:
            lines.append(f"  {p} -> <error: {exc}>")
    return "\n".join(lines)

def _extract_comp_archives(comp_roots, tmp_extract_dir: str, data_dir: str) -> bool:
    """Extract competition-folder .zip/.tar archives (e.g. train.zip + test.zip) into /tmp and resolve."""
    archives = []
    csv_dirs = []
    for root_base in comp_roots:
        if not os.path.isdir(root_base):
            continue
        base_depth = root_base.rstrip(os.sep).count(os.sep)
        for cur, dirs, files in os.walk(root_base, followlinks=True):
            dirs[:] = [d for d in dirs if d not in _SKIP_WALK_DIRS]
            if cur.rstrip(os.sep).count(os.sep) - base_depth > 2:
                dirs[:] = []
                continue
            if "train.csv" in files and "test.csv" in files:
                csv_dirs.append(cur)
            for fn in sorted(files):
                low = fn.lower()
                if fn.startswith("task4_colab_pack"):
                    continue
                if low.endswith((".zip", ".tar", ".tar.gz", ".tgz")):
                    fp = os.path.join(cur, fn)
                    if os.path.getsize(fp) >= 100_000:
                        archives.append((cur, fp))
    if not archives:
        return False

    print(f"Extracting {len(archives)} competition archive(s) -> {tmp_extract_dir} ...")
    os.makedirs(tmp_extract_dir, exist_ok=True)
    for _, fp in archives:
        low = fp.lower()
        if low.endswith(".zip"):
            with zipfile.ZipFile(fp, "r") as zf:
                zf.extractall(tmp_extract_dir)
        else:
            subprocess.check_call(["tar", "--exclude=._*", "--exclude=*/._*", "-xf", fp, "-C", tmp_extract_dir])

    for cd in csv_dirs:
        for fn in ("train.csv", "test.csv", "folds_sgkf.npy"):
            src_f = os.path.join(cd, fn)
            dst_f = os.path.join(tmp_extract_dir, fn)
            if os.path.isfile(src_f) and not os.path.isfile(dst_f):
                shutil.copy2(src_f, dst_f)

    layout = _find_competition_data_dir([tmp_extract_dir], direct_dirs=(tmp_extract_dir,))
    if layout is not None:
        csv_dir, eff_tr, eff_te, cls_sub = layout
        prepare_competition_dir(csv_dir, data_dir, tr_dir=eff_tr, te_dir=eff_te, class_subfolders=cls_sub)
        return True
    return False

def _download_official_gdrive_fallback(data_dir: str, fallback_csv_dirs) -> bool:
    """Download official Task 4 dataset archive from Google Drive via gdown when Kaggle input lacks JPGs."""
    base_tmp = os.path.dirname(os.path.abspath(data_dir))
    dl_zip = os.path.join(base_tmp, "task4_official_gdrive.zip")
    raw_dir = os.path.join(base_tmp, "t4_official_raw")
    try:
        try:
            gdown = importlib.import_module("gdown")
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            gdown = importlib.import_module("gdown")
        print(f"Downloading official Task 4 dataset (Google Drive id={OFFICIAL_GDRIVE_FILE_ID}) -> {dl_zip} ...", flush=True)
        url = f"https://drive.google.com/uc?id={OFFICIAL_GDRIVE_FILE_ID}"
        out_p = gdown.download(url, dl_zip, quiet=False)
        if not out_p or not os.path.isfile(dl_zip) or os.path.getsize(dl_zip) < 10_000_000:
            return False
        os.makedirs(raw_dir, exist_ok=True)
        print(f"Extracting official archive -> {raw_dir} ...", flush=True)
        if zipfile.is_zipfile(dl_zip):
            with zipfile.ZipFile(dl_zip, "r") as zf:
                zf.extractall(raw_dir)
        else:
            subprocess.check_call(["tar", "--exclude=._*", "--exclude=*/._*", "-xf", dl_zip, "-C", raw_dir])
        if os.path.isfile(dl_zip):
            os.remove(dl_zip)

        # Copy CSVs from existing competition directory OR materialize canonical embedded train/test CSVs (Colab-ready)
        if not (os.path.isfile(os.path.join(raw_dir, "train.csv")) and os.path.isfile(os.path.join(raw_dir, "test.csv"))):
            copied_csv = False
            for cd in fallback_csv_dirs:
                if os.path.isfile(os.path.join(cd, "train.csv")) and os.path.isfile(os.path.join(cd, "test.csv")):
                    shutil.copy2(os.path.join(cd, "train.csv"), os.path.join(raw_dir, "train.csv"))
                    shutil.copy2(os.path.join(cd, "test.csv"), os.path.join(raw_dir, "test.csv"))
                    copied_csv = True
                    break
            if not copied_csv:
                decode_embedded_train_csv().to_csv(os.path.join(raw_dir, "train.csv"), index=False)
                decode_embedded_test_csv().to_csv(os.path.join(raw_dir, "test.csv"), index=False)
                print("Wrote canonical embedded train.csv & test.csv for Google Drive archive (zero upload needed!)", flush=True)

        layout = _find_competition_data_dir([raw_dir], direct_dirs=(raw_dir, os.path.join(raw_dir, "task4"), os.path.join(raw_dir, "data")))
        if layout is None:
            _tr_probe, _te_probe = decode_embedded_train_csv(), decode_embedded_test_csv()
            for _cur, _dirs, _files in os.walk(raw_dir):
                _dl = {d.lower(): os.path.join(_cur, d) for d in _dirs}
                if "train" in _dl and "test" in _dl:
                    _pr = _probe_pair(_dl["train"], _dl["test"], _tr_probe, _te_probe)
                    if _pr is not None:
                        _eff_tr, _eff_te, _cls_sub = _pr
                        _csv_tgt = _cur if (os.path.isfile(os.path.join(_cur, "train.csv")) and os.path.isfile(os.path.join(_cur, "test.csv"))) else raw_dir
                        if not os.path.isfile(os.path.join(_csv_tgt, "train.csv")):
                            _tr_probe.to_csv(os.path.join(_csv_tgt, "train.csv"), index=False)
                            _te_probe.to_csv(os.path.join(_csv_tgt, "test.csv"), index=False)
                        layout = (_csv_tgt, _eff_tr, _eff_te, _cls_sub)
                        break
        if layout is None:
            return False
        csv_dir, eff_tr, eff_te, cls_sub = layout
        prepare_competition_dir(csv_dir, data_dir, tr_dir=eff_tr, te_dir=eff_te, class_subfolders=cls_sub)
        return True
    except Exception as exc:
        print(f"[GDrive Fallback Warning] {exc}")
        if os.path.isfile(dl_zip):
            os.remove(dl_zip)
        return False

def prepare_dataset_pack(
    data_dir: str = DATA_DIR,
    search_roots=None,
    direct_comp_dirs=KAGGLE_DIRECT_COMP_DIRS,
    allow_gdrive_fallback: bool = AUTO_GDOWN_OFFICIAL,
) -> None:
    if is_data_ready(data_dir):
        print("Dataset already ready and verified in", data_dir)
        return
    os.makedirs(data_dir, exist_ok=True)
    if search_roots is None:
        search_roots = ["/content"] if IS_COLAB else ["/kaggle/input", "/kaggle/working", "/content", "."]
    eff_direct_dirs = () if IS_COLAB else direct_comp_dirs
    snapshot = ""

    if IS_COLAB:
        print("[Colab Fast-Path] Checking /content for local Task 4 pack or downloading official archive (~626 MB, ~30-60s)...", flush=True)
    else:
        # 1. O(1) Kaggle competition fast-path (supports flat, nested, class subfolders)
        for dpath in eff_direct_dirs:
            layout = _resolve_comp_layout(dpath)
            if layout is not None:
                csv_dir, eff_tr, eff_te, cls_sub = layout
                print(f"Fast-path Kaggle dataset -> csv={csv_dir} | train={eff_tr} | test={eff_te} | cls_sub={cls_sub}")
                prepare_competition_dir(csv_dir, data_dir, tr_dir=eff_tr, te_dir=eff_te, class_subfolders=cls_sub)
                print(f"Normalized symlinks + verified SGKF folds into {data_dir} (0 GB working quota used!)")
                return

        diag_roots = list(eff_direct_dirs) + [
            "/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main",
            "/kaggle/input/competitions/player-intelligence-system-cpe-342-ml",
            "/kaggle/input/player-intelligence-system-cpe-342-ml",
            "/kaggle/input",
        ]
        snapshot = _collect_dir_snapshot(diag_roots)
        if snapshot:
            print("Inspecting attached input layout:\n" + snapshot)

        # 2. Local archive (.zip/.tar) inside 04_image_classification / Task 4 folder -> extract to /tmp
        tmp_extract = os.path.join(os.path.dirname(os.path.abspath(data_dir)), "t4_comp_extract")
        if _extract_comp_archives(list(eff_direct_dirs), tmp_extract, data_dir):
            print(f"Normalized extracted competition archives + verified SGKF folds -> {data_dir}")
            return

    # 3. Discover task4_colab_pack archive or split parts (pruned os.walk)
    parts_dirs = []
    direct_candidates = []
    for rb in search_roots:
        if not os.path.isdir(rb):
            continue
        rb_depth = rb.rstrip(os.sep).count(os.sep)
        for cur, dirs, files in os.walk(rb, followlinks=True):
            dirs[:] = [d for d in dirs if d not in _SKIP_WALK_DIRS]
            if cur.rstrip(os.sep).count(os.sep) - rb_depth > 4:
                dirs[:] = []
                continue
            if os.path.basename(cur) == "task4_colab_pack_parts" and glob.glob(os.path.join(cur, "part_*")):
                parts_dirs.append(cur)
            for fn in ("task4_colab_pack.tar", "task4_colab_pack.zip"):
                if fn in files:
                    direct_candidates.append(os.path.join(cur, fn))

    merged_tar = os.path.join(os.path.dirname(os.path.abspath(data_dir)), "task4_colab_pack_joined.tar")
    joined_from_parts = False
    parts_error = None
    pdir = parts_dirs[0] if parts_dirs else None
    if pdir and not any(os.path.isfile(c) for c in direct_candidates):
        expected_paths, bad_parts = [], []
        for idx in range(EXPECTED_NUM_PARTS):
            pname = f"part_{idx:02d}"
            exp_sz = EXPECTED_LAST_PART_BYTES if idx == EXPECTED_NUM_PARTS - 1 else EXPECTED_STD_PART_BYTES
            ppath = os.path.join(pdir, pname)
            if not os.path.isfile(ppath):
                bad_parts.append(f"{pname} (missing)")
            elif os.path.getsize(ppath) != exp_sz:
                bad_parts.append(f"{pname} ({os.path.getsize(ppath):,}B != {exp_sz:,}B)")
            else:
                expected_paths.append(ppath)
        if bad_parts:
            parts_error = f"Incomplete pack parts in {pdir}: {bad_parts[:8]}"
        else:
            print(f"Joining {len(expected_paths)} verified chunks from {pdir} -> {merged_tar} ...")
            h = hashlib.sha256()
            total = 0
            with open(merged_tar, "wb") as out_f:
                for pf in expected_paths:
                    with open(pf, "rb") as in_f:
                        while True:
                            buf = in_f.read(8 * 1024 * 1024)
                            if not buf:
                                break
                            out_f.write(buf)
                            h.update(buf)
                            total += len(buf)
            if total != EXPECTED_TAR_BYTES or h.hexdigest() != EXPECTED_TAR_SHA256:
                os.remove(merged_tar)
                raise RuntimeError(f"Joined archive checksum mismatch: bytes={total}, sha256={h.hexdigest()}")
            joined_from_parts = True
            direct_candidates.insert(0, merged_tar)
            print(f"Verified pack SHA-256: {EXPECTED_TAR_SHA256[:16]}... ({total / (1024**2):.1f} MiB)")

    pack_path = next((p for p in direct_candidates if os.path.isfile(p) and (p.endswith(".zip") or os.path.getsize(p) >= 700_000_000)), None)
    if pack_path is not None:
        print("Extracting", pack_path, "->", data_dir, "...")
        if pack_path.endswith(".zip"):
            with zipfile.ZipFile(pack_path, "r") as zf:
                zf.extractall(data_dir)
        else:
            subprocess.check_call(["tar", "--exclude=._*", "--exclude=*/._*", "-xf", pack_path, "-C", data_dir])
        if joined_from_parts and os.path.isfile(merged_tar):
            os.remove(merged_tar)
        if not is_data_ready(data_dir):
            layout = _resolve_comp_layout(data_dir)
            if layout is not None:
                csv_dir, eff_tr, eff_te, cls_sub = layout
                prepare_competition_dir(csv_dir, data_dir, tr_dir=eff_tr, te_dir=eff_te, class_subfolders=cls_sub)
        return

    # 4. General pruned competition layout walk fallback
    layout = _find_competition_data_dir(search_roots, direct_dirs=eff_direct_dirs)
    if layout is not None:
        csv_dir, eff_tr, eff_te, cls_sub = layout
        print(f"Detected Kaggle/Competition Task 4 dataset -> csv={csv_dir} | train={eff_tr} | cls_sub={cls_sub}")
        prepare_competition_dir(csv_dir, data_dir, tr_dir=eff_tr, te_dir=eff_te, class_subfolders=cls_sub)
        print(f"Normalized symlinks + verified SGKF folds into {data_dir} (0 GB working quota used!)")
        return

    # 5. Official Google Drive fallback (579 MB) if competition input did not include extracted JPGs
    if allow_gdrive_fallback and _download_official_gdrive_fallback(data_dir, eff_direct_dirs):
        print(f"Normalized official Google Drive dataset + verified SGKF folds into {data_dir}")
        return

    raise RuntimeError(
        parts_error
        or f"Task 4 dataset not found! Add Competition 'player-intelligence-system-cpe-342-ml' (or turn Internet ON for Drive fallback).\nScanned:\n{snapshot}"
    )

if not os.environ.get("T4_SKIP_AUTO_PREP"):
    prepare_dataset_pack()
    assert is_data_ready(DATA_DIR), "Dataset setup incomplete!"
    _tr_ok = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
    _te_ok = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
    print(f"Ready ({DATA_DIR})! Train rows={len(_tr_ok):,} | Test rows={len(_te_ok):,} | O(1) image checks passed")

In [ ]:
# Cell 3: Fast Vectorized De-Corruption v4 (OpenCV + 12-vCPU + SSD Cache), ExtHard & JPEG Preloader
import io, os, pickle, time
from concurrent.futures import ThreadPoolExecutor
import multiprocessing as mp
import numpy as np
from PIL import Image, ImageFilter

try:
    import cv2
    _HAS_CV2 = True
except ImportError:
    _HAS_CV2 = False

def detect_angle_fast_radon(im_rgb: Image.Image) -> float:
    """Vectorized border rotation detector via BoxBlur(2) adaptive threshold + Theil-Sen median slope (R^2 >= 0.85)."""
    g = np.asarray(im_rgb.convert("L").filter(ImageFilter.BoxBlur(2)), dtype=np.float32)
    h, w = g.shape
    corner_bg = np.percentile(
        np.concatenate([g[:6, :6].ravel(), g[:6, -6:].ravel(), g[-6:, :6].ravel(), g[-6:, -6:].ravel()]),
        25,
    )
    interior_med = float(np.median(g[h // 4 : 3 * h // 4, w // 4 : 3 * w // 4]))
    if interior_med - corner_bg < 10.0:
        return 0.0
    thr = corner_bg + max(8.0, 0.28 * (interior_med - corner_bg))
    mask = g[:, 4 : w - 4 : 4] >= thr
    xs_grid = np.arange(4, w - 4, 4, dtype=np.float64)
    has_any = mask.any(axis=0)
    if has_any.sum() < 6:
        return 0.0

    best_score, best_ang = 0.0, 0.0
    top_ys = mask.argmax(axis=0)
    bot_ys = (h - 1) - mask[::-1].argmax(axis=0)
    for ys_raw in (top_ys, bot_ys):
        valid = has_any & (ys_raw >= 2) & (ys_raw <= h - 3)
        if valid.sum() < 6:
            continue
        ex_a, ey_a = xs_grid[valid], ys_raw[valid].astype(np.float64)
        splits = np.where(np.diff(ex_a) > 12)[0] + 1
        for seg in np.split(np.arange(len(ex_a)), splits):
            if len(seg) < 6:
                continue
            sx, sy = ex_a[seg], ey_a[seg]
            dx = sx[:, None] - sx[None, :]
            dy = sy[:, None] - sy[None, :]
            vp = np.abs(dx) >= 12
            if not vp.any():
                continue
            s_med = float(np.median(dy[vp] / dx[vp]))
            b_med = float(np.median(sy - s_med * sx))
            inl = np.abs(sy - (s_med * sx + b_med)) < 2.8
            n_inl = int(inl.sum())
            if n_inl >= 6:
                sxi, syi = sx[inl], sy[inl]
                xm, ym = sxi.mean(), syi.mean()
                denom = float(np.sum((sxi - xm) ** 2)) + 1e-6
                s_ref = float(np.sum((sxi - xm) * (syi - ym)) / denom)
                ang = float(np.degrees(np.arctan(s_ref)))
                pred_y = s_ref * (sxi - xm) + ym
                ss_res = float(np.sum((syi - pred_y) ** 2))
                ss_tot = float(np.sum((syi - ym) ** 2)) + 1e-6
                r2 = 1.0 - ss_res / ss_tot
                if 2.0 <= abs(ang) <= 32.0 and r2 >= 0.85 and n_inl * r2 > best_score:
                    best_score, best_ang = n_inl * r2, ang
    return best_ang

def decorrupt_pil_v4(im: Image.Image) -> Image.Image:
    """Fast deterministic derotation, crop, YCbCr chroma denoise (OpenCV/PIL), and vectorized capped 2.2x stretch."""
    im = im.convert("RGB")
    w0, h0 = im.size
    ang = detect_angle_fast_radon(im)
    im_rot = im.rotate(ang, resample=Image.BICUBIC, fillcolor=(0, 0, 0)) if abs(ang) >= 2.0 else im

    g = np.asarray(im_rot.convert("L").filter(ImageFilter.BoxBlur(2)), dtype=np.float32)
    cbg = np.percentile(
        np.concatenate([g[:5, :5].ravel(), g[:5, -5:].ravel(), g[-5:, :5].ravel(), g[-5:, -5:].ravel()]),
        25,
    )
    imed = float(np.median(g[h0 // 4 : 3 * h0 // 4, w0 // 4 : 3 * w0 // 4]))
    thr = cbg + max(6.0, 0.22 * max(imed - cbg, 0.0))
    vm = g >= thr
    ys = np.where(vm.mean(axis=1) >= 0.55)[0]
    xs = np.where(vm.mean(axis=0) >= 0.55)[0]
    if len(ys) >= int(h0 * 0.58) and len(xs) >= int(w0 * 0.58):
        im_crop = im_rot.crop((int(xs[0]), int(ys[0]), int(xs[-1]) + 1, int(ys[-1]) + 1)).resize((w0, h0), Image.BICUBIC)
    else:
        im_crop = im_rot

    if _HAS_CV2:
        ycrcb = cv2.cvtColor(np.asarray(im_crop, dtype=np.uint8), cv2.COLOR_RGB2YCrCb)
        y_c = cv2.medianBlur(ycrcb[:, :, 0], 3)
        cr_c = cv2.GaussianBlur(ycrcb[:, :, 1], (0, 0), 1.5)
        cb_c = cv2.GaussianBlur(ycrcb[:, :, 2], (0, 0), 1.5)
        da = cv2.cvtColor(np.stack([y_c, cr_c, cb_c], axis=2), cv2.COLOR_YCrCb2RGB).astype(np.float32)
    else:
        y_ch, cb_ch, cr_ch = im_crop.convert("YCbCr").split()
        im_dn = Image.merge(
            "YCbCr",
            (
                y_ch.filter(ImageFilter.MedianFilter(size=3)),
                cb_ch.filter(ImageFilter.GaussianBlur(radius=1.5)),
                cr_ch.filter(ImageFilter.GaussianBlur(radius=1.5)),
            ),
        ).convert("RGB")
        da = np.asarray(im_dn, dtype=np.float32)

    mh0, mh1 = max(1, h0 // 16), h0 - max(1, h0 // 16)
    mw0, mw1 = max(1, w0 // 16), w0 - max(1, w0 // 16)
    sub = da[mh0:mh1, mw0:mw1]
    nz = sub.mean(axis=2) >= 8.0
    out = da
    if nz.sum() > 200:
        los, his = np.percentile(sub[nz], [2.0, 98.0], axis=0)
        spans = np.maximum(his - los, 255.0 / 2.2)
        mids = 0.5 * (los + his)
        stretched = (da - (mids - 0.5 * spans)[None, None, :]) / spans[None, None, :] * 235.0 + 10.0
        out = np.clip(0.80 * stretched + 0.20 * da, 0.0, 255.0)
    return Image.fromarray(out.astype(np.uint8))

def _decorrupt_to_jpeg(item) -> bytes:
    im = Image.open(io.BytesIO(item)).convert("RGB") if isinstance(item, (bytes, bytearray)) else Image.fromarray(item)
    cl = decorrupt_pil_v4(im)
    buf = io.BytesIO()
    cl.save(buf, format="JPEG", quality=95)
    return buf.getvalue()

def batch_precompute_decorrupt_bytes(items, label: str = "split", cache_path: str = None):
    """Precompute decorrupt_pil_v4 -> JPEG q95 bytes ONCE across CPU workers with optional local SSD cache."""
    t0 = time.time()
    if cache_path and os.path.isfile(cache_path):
        try:
            with open(cache_path, "rb") as f:
                cached = pickle.load(f)
            if len(cached) == len(items):
                print(f"[CACHE HIT] decorrupt_v4 {label} ({len(cached):,}) loaded in {time.time()-t0:.1f}s -> {cache_path}", flush=True)
                return cached
        except Exception:
            pass
    n_cpu = min(12, max(1, os.cpu_count() or 2))
    out = None
    if n_cpu >= 2 and os.name == "posix":
        try:
            ctx = mp.get_context("fork")
            with ctx.Pool(processes=n_cpu) as pool:
                out = pool.map(_decorrupt_to_jpeg, items, chunksize=256)
        except Exception:
            out = None
    if out is None:
        with ThreadPoolExecutor(max_workers=n_cpu) as pool:
            out = list(pool.map(_decorrupt_to_jpeg, items))
    if cache_path:
        try:
            tmp_p = f"{cache_path}.tmp.{os.getpid()}"
            with open(tmp_p, "wb") as f:
                pickle.dump(out, f, protocol=pickle.HIGHEST_PROTOCOL)
            os.replace(tmp_p, cache_path)
        except Exception:
            pass
    print(f"Precomputed decorrupt_v4 JPEG bytes for {label} ({len(out):,}, workers={n_cpu}, cv2={_HAS_CV2}) in {time.time()-t0:.1f}s", flush=True)
    return out

def make_deterministic_exthard(jpeg_bytes_list, indices: np.ndarray, seed: int = 123):
    """Generate deterministic ExtremeHard validation images matching kaggle-t4-sgkf-hardened.ipynb parity."""
    rng = np.random.default_rng(seed)
    out = []
    for k, idx in enumerate(indices):
        im = Image.open(io.BytesIO(jpeg_bytes_list[int(idx)])).convert("RGB")
        im = im.rotate(float(rng.uniform(-35.0, 35.0)), resample=Image.BILINEAR, fillcolor=(0, 0, 0))
        arr = np.asarray(im, dtype=np.float32) * (1.0 / 255.0)
        mode = k % 4
        if mode == 0:
            arr = np.power(np.clip(arr, 1e-4, 1.0), float(rng.uniform(1.8, 2.8))) * float(rng.uniform(0.35, 0.65))
        elif mode == 1:
            arr = np.clip(arr * float(rng.uniform(1.3, 2.0)) + float(rng.uniform(0.30, 0.50)), 0.0, 1.0)
        elif mode == 2:
            g = rng.uniform(0.35, 2.0, size=(1, 1, 3)).astype(np.float32)
            g /= float(g.max())
            arr = np.clip(np.power(arr, 0.85) * g * float(rng.uniform(0.8, 1.4)), 0.0, 1.0)
        else:
            arr = np.clip(0.5 + (arr - 0.5) * float(rng.uniform(0.4, 0.7)) + rng.uniform(-0.15, 0.15, (1, 1, 3)), 0.0, 1.0)
        arr = arr + rng.normal(0.0, float(rng.uniform(0.03, 0.07)), size=arr.shape).astype(np.float32)
        out.append(np.clip(arr * 255.0, 0.0, 255.0).astype(np.uint8))
    return out

def load_jpeg_bytes(folder: str, filenames):
    """Preload compressed JPEG buffers into host RAM."""
    buffers = []
    for fn in filenames:
        with open(os.path.join(folder, fn), "rb") as f:
            buffers.append(f.read())
    return buffers

print(f"Fast De-corruption v4 (cv2={_HAS_CV2}), parallel precompute, ExtHard generator & JPEG preloader ready.")


In [ ]:
# Cell 4: Stage 1 Multi-Foundation 6-Backbone x 6-View Frozen Feature Extractor (A100 High-Res & Atomic Cached)
import gc, os, threading, time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

tr_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
te_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
if "file_name" not in tr_df.columns:
    tr_df["file_name"] = tr_df["id"].astype(str) + ".jpg"
if "file_name" not in te_df.columns:
    te_df["file_name"] = te_df["id"].astype(str) + ".jpg"

CLASS_MAP = {"7-11k": 0, "7-11K": 0, "fiveN": 1, "freefried": 2, "pube": 3, "roblock": 4}
y = tr_df["label"].astype(int).to_numpy() if "label" in tr_df.columns else tr_df["class"].map(CLASS_MAP).astype(int).to_numpy()
folds = np.load(os.path.join(DATA_DIR, "folds_sgkf.npy")).astype(int)
assert len(tr_df) == len(folds) == len(y)

print("Preloading compressed JPEG bytes into RAM...")
t_load = time.time()
tr_bytes = load_jpeg_bytes(os.path.join(DATA_DIR, "train"), tr_df["file_name"].tolist())
te_bytes = load_jpeg_bytes(os.path.join(DATA_DIR, "test"), te_df["file_name"].tolist())
print(f"Preloaded train={len(tr_bytes):,}, test={len(te_bytes):,} images in {time.time()-t_load:.1f}s")

# Build deterministic fold-matched ExtHard array matching kaggle-t4-sgkf-hardened.ipynb seed=123+k
ext_all_u8 = [None] * len(tr_bytes)
for k in range(5):
    va_k = np.where(folds == k)[0]
    ext_k = make_deterministic_exthard(tr_bytes, va_k, seed=123 + k)
    for idx_val, img_u8 in zip(va_k, ext_k):
        ext_all_u8[int(idx_val)] = img_u8
assert all(x is not None for x in ext_all_u8)

@dataclass(frozen=True)
class FoundSpec:
    key: str
    candidates: Tuple[str, ...]
    target_hw: Tuple[int, int]
    batch_size: int

_HW_P14 = (224, 448) if HIGH_RES_A100 else (196, 392)
_HW_P16 = (224, 448) if HIGH_RES_A100 else (192, 384)
FOUND_SPECS: List[FoundSpec] = [
    FoundSpec(
        "siglip2_so400m",
        (
            "vit_so400m_patch16_siglip2_384",
            "vit_so400m_patch14_siglip_384",
            "vit_so400m_patch14_siglip_384.webli",
            "vit_large_patch16_siglip_384",
        ),
        _HW_P16,
        36,
    ),
    FoundSpec(
        "dinov3_vitl",
        ("vit_large_patch16_dinov3.lvd1689m", "vit_large_patch14_reg4_dinov2.lvd142m", "vit_large_patch14_dinov2.lvd142m"),
        _HW_P16,
        48,
    ),
    FoundSpec(
        "dinov2_vitg14",
        ("vit_large_patch14_reg4_dinov2.lvd142m", "vit_large_patch14_dinov2.lvd142m")
        if FAST_T4_TIER
        else ("vit_giant_patch14_reg4_dinov2.lvd142m", "vit_giant_patch14_dinov2.lvd142m", "vit_large_patch14_reg4_dinov2.lvd142m"),
        _HW_P14,
        48 if FAST_T4_TIER else 28,
    ),
    FoundSpec(
        "eva02_large",
        ("eva02_large_patch14_448.mim_m38m_ft_in22k_in1k", "eva02_large_patch14_448.mim_m38m_ft_in22k"),
        _HW_P14,
        48,
    ),
    FoundSpec(
        "clip_cnxl",
        ("convnext_large_mlp.clip_laion2b_soup_ft_in12k_in1k_384", "convnext_large_mlp.clip_laion2b_soup_ft_in12k_in1k_320")
        if FAST_T4_TIER
        else (
            "convnext_xxlarge.clip_laion2b_soup_ft_in12k",
            "convnext_large_mlp.clip_laion2b_soup_ft_in12k_in1k_384",
            "convnext_large_mlp.clip_laion2b_soup_ft_in12k_in1k_320",
        ),
        _HW_P16,
        48 if FAST_T4_TIER else 36,
    ),
    FoundSpec(
        "clip_vith14",
        ("vit_large_patch14_clip_336.openai", "vit_large_patch14_clip_224.openai")
        if FAST_T4_TIER
        else (
            "vit_huge_patch14_clip_378.dfn5b",
            "vit_large_patch14_clip_336.openai",
            "vit_huge_patch14_clip_224.dfn5b",
            "vit_large_patch14_clip_224.openai",
        ),
        _HW_P14,
        48 if FAST_T4_TIER else 36,
    ),
]

def _pick_hw_for_model(mname: str, default_hw: Tuple[int, int]) -> Tuple[int, int]:
    """Select exact 2:1 patch-aligned resolution (224x448 on A100 is divisible by 14, 16, and 32)."""
    if HIGH_RES_A100:
        return (224, 448)
    if "patch14" in mname:
        return (196, 392)
    return (192, 384) if ("patch16" in mname or "convnext" in mname or "dinov3" in mname) else default_hw

def _center_crop_resize(x: torch.Tensor, frac: float = 0.82) -> torch.Tensor:
    """Zoom into interior frac x frac HUD/subject region on GPU to reduce edge border distortion."""
    _, _, h, w = x.shape
    ch, cw = max(8, int(round(h * frac))), max(8, int(round(w * frac)))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = x[:, :, y0 : y0 + ch, x0 : x0 + cw]
    return F.interpolate(crop, size=(h, w), mode="bicubic", align_corners=False).clamp(0.0, 1.0)

need_stage1_extract = RUN_STAGE1_FOUNDATION_PROBES and any(
    not os.path.isfile(os.path.join(OUT_DIR, f"emb_v{N_VIEWS}_{fs.key}.npz")) for fs in FOUND_SPECS
)
if need_stage1_extract or DO_STAGE2:
    tr_dec_bytes = batch_precompute_decorrupt_bytes(tr_bytes, "train", cache_path=os.path.join(DATA_DIR, "dec_v4_train.pkl"))
    ext_dec_bytes = batch_precompute_decorrupt_bytes(ext_all_u8, "exthard", cache_path=os.path.join(DATA_DIR, "dec_v4_exthard.pkl"))
    te_dec_bytes = batch_precompute_decorrupt_bytes(te_bytes, "test", cache_path=os.path.join(DATA_DIR, "dec_v4_test.pkl"))
else:
    tr_dec_bytes, ext_dec_bytes, te_dec_bytes = None, None, None

def atomic_savez_compressed(path: str, **arrays):
    tmp_path = f"{path}.tmp.{os.getpid()}.{threading.get_ident()}.npz"
    np.savez_compressed(tmp_path, **arrays)
    os.replace(tmp_path, path)

def create_timm_eval_backbone(candidates: Tuple[str, ...], target_hw: Tuple[int, int], pretrained: bool = True):
    """Load backbone with num_classes=0 and auto-detect native/2:1 img_size."""
    last_exc = None
    if pretrained:
        avail = set(timm.list_models("*"))
        ordered = [c for c in candidates if c in avail] + [c for c in candidates if c not in avail]
    else:
        ordered = list(candidates)
    for mname in ordered:
        pref_hw = _pick_hw_for_model(mname, target_hw)
        for try_hw in (True, False):
            try:
                kw = {"pretrained": pretrained, "num_classes": 0}
                if try_hw and ("vit" in mname or "eva" in mname or "siglip" in mname):
                    kw["img_size"] = pref_hw
                m = timm.create_model(mname, **kw)
                cfg = timm.data.resolve_model_data_config(m)
                eff_hw = tuple(pref_hw) if (try_hw or "convnext" in mname) else tuple(cfg["input_size"][1:])
                return mname, m, tuple(cfg["mean"]), tuple(cfg["std"]), eff_hw, try_hw
            except Exception as exc:
                last_exc = exc
    raise RuntimeError(f"Could not instantiate any candidate from {candidates}: {last_exc}") from last_exc

class FourViewPrecomputedDataset(Dataset):
    """Fast decode of [raw, precomputed_decorrupt_v4]; flips, crops & norm run vectorized on GPU."""
    def __init__(self, raw_items, dec_bytes, indices, eff_hw: Tuple[int, int], is_bytes: bool):
        self.raw_items = raw_items
        self.dec_bytes = dec_bytes
        self.indices = indices
        self.wh = (eff_hw[1], eff_hw[0])
        self.is_bytes = is_bytes

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, pos: int):
        ri = int(self.indices[pos])
        obj = self.raw_items[ri]
        im_raw = Image.open(io.BytesIO(obj)).convert("RGB") if self.is_bytes else Image.fromarray(obj)
        im_dec = Image.open(io.BytesIO(self.dec_bytes[ri])).convert("RGB")
        if im_raw.size != self.wh:
            im_raw = im_raw.resize(self.wh, Image.BICUBIC)
        if im_dec.size != self.wh:
            im_dec = im_dec.resize(self.wh, Image.BICUBIC)
        pair = np.stack([np.asarray(im_raw, dtype=np.uint8), np.asarray(im_dec, dtype=np.uint8)], axis=0)
        return torch.from_numpy(pair).permute(0, 3, 1, 2).contiguous()

def _extract_split_on_device(model, mean_t, std_t, raw_items, dec_bytes, indices, eff_hw, bs: int, is_bytes: bool, device: torch.device, amp_dtype, workers: int, split_name: str = "", n_views: int = 6):
    div_v = 2 if n_views <= 4 else 3
    dl_bs = max(16, min(64, int(bs) // div_v)) if len(indices) >= 16 else max(1, min(len(indices), int(bs)))
    ds = FourViewPrecomputedDataset(raw_items, dec_bytes, indices, eff_hw, is_bytes=is_bytes)
    dl_kw = {"batch_size": dl_bs, "shuffle": False, "num_workers": workers, "pin_memory": (device.type == "cuda" and workers > 0)}
    if workers > 0:
        dl_kw["prefetch_factor"] = 2
    dl = DataLoader(ds, **dl_kw)

    cur_bs = int(bs)
    raw2_out, all_out = [], []
    n_total = len(indices)
    done_n = 0
    t_sp = time.time()
    last_log = t_sp
    lbl = f"[{split_name}] " if split_name else ""
    eff_v = 4 if n_views <= 4 else 6
    with torch.inference_mode():
        for pair_u8 in dl:
            bsz = pair_u8.shape[0]
            pair_f = pair_u8.to(device, non_blocking=(device.type == "cuda" and workers > 0)).float() * (1.0 / 255.0)
            x_raw = pair_f[:, 0]
            x_dec = pair_f[:, 1]
            vlist = [x_raw, torch.flip(x_raw, dims=[-1]), x_dec, torch.flip(x_dec, dims=[-1])]
            if eff_v == 6:
                vlist.extend([_center_crop_resize(x_raw, 0.82), _center_crop_resize(x_dec, 0.82)])
            views = torch.cat(vlist, dim=0)
            chunks = []
            s = 0
            while s < len(views):
                xb = (views[s : s + cur_bs] - mean_t) / std_t
                try:
                    with torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.inference_mode():
                        ft = model(xb)
                    if ft.ndim > 2:
                        ft = ft.mean(dim=tuple(range(2, ft.ndim))) if ft.ndim == 4 else ft.mean(dim=1)
                    chunks.append(F.normalize(ft.float(), p=2, dim=-1))
                    s += len(xb)
                except RuntimeError as exc:
                    if "out of memory" not in str(exc).lower() or cur_bs <= 8:
                        raise
                    torch.cuda.empty_cache()
                    cur_bs = max(8, cur_bs // 2)
            emb = torch.cat(chunks, dim=0).view(eff_v, bsz, -1)
            e_r2 = F.normalize(0.5 * (emb[0] + emb[1]), p=2, dim=-1)
            if eff_v == 4:
                e_a = F.normalize(0.25 * (emb[0] + emb[1] + emb[2] + emb[3]), p=2, dim=-1)
            else:
                e_a = F.normalize(
                    0.20 * emb[0] + 0.20 * emb[1] + 0.20 * emb[2] + 0.20 * emb[3] + 0.10 * emb[4] + 0.10 * emb[5],
                    p=2,
                    dim=-1,
                )
            raw2_out.append(e_r2.cpu().numpy().astype(np.float16))
            all_out.append(e_a.cpu().numpy().astype(np.float16))
            done_n += bsz
            now = time.time()
            if split_name and (now - last_log >= 15.0 or done_n == n_total):
                dt = max(now - t_sp, 1e-3)
                ips = done_n / dt
                eta = (n_total - done_n) / max(ips, 1e-3)
                print(f"    {lbl}{done_n:5d}/{n_total:,} ({100.0*done_n/n_total:5.1f}%) | {ips:5.1f} img/s ({ips*eff_v:6.1f} views/s) | ETA {eta:4.0f}s", flush=True)
                last_log = now
    return np.concatenate(raw2_out, axis=0), np.concatenate(all_out, axis=0)

def extract_backbone_embeddings(fspec: FoundSpec):
    out_npz = os.path.join(OUT_DIR, f"emb_v{N_VIEWS}_{fspec.key}.npz")
    if os.path.isfile(out_npz):
        try:
            with np.load(out_npz) as z:
                if len(z["tr_all4"]) == len(tr_bytes) and len(z["te_all4"]) == len(te_bytes):
                    print(f"[CACHE HIT] {fspec.key}: {z['tr_all4'].shape} ({out_npz})")
                    return out_npz
        except Exception:
            pass

    t0 = time.time()
    eff_bs = max(16, int(round(fspec.batch_size * EXTRACT_BS_MULT)))
    mname, base_model, mean_tup, std_tup, eff_hw, used_hw = create_timm_eval_backbone(fspec.candidates, fspec.target_hw, pretrained=True)
    print(f"\n>>> Extracting {N_VIEWS}-view [{fspec.key}] using {mname} @ {eff_hw} | eff_bs={eff_bs} (VRAM={GPU_VRAM_GB:.1f}GB, GPUs={N_GPUS})...", flush=True)

    use_dual = ENABLE_DUAL_GPU_PARALLEL and N_GPUS >= 2 and DEVICE.type == "cuda"
    splits = [
        ("tr", tr_bytes, tr_dec_bytes, np.arange(len(tr_bytes)), True),
        ("ext", ext_all_u8, ext_dec_bytes, np.arange(len(ext_all_u8)), False),
        ("te", te_bytes, te_dec_bytes, np.arange(len(te_bytes)), True),
    ]
    results = {}

    if use_dual:
        sd_cpu = {k: v.cpu() for k, v in base_model.state_dict().items()}
        del base_model
        gc.collect()

        def _worker(gid: int):
            dev_g = torch.device(f"cuda:{gid}")
            torch.cuda.set_device(dev_g)
            amp_g, _ = select_amp_dtype(dev_g)
            kw_g = {"pretrained": False, "num_classes": 0}
            if used_hw and ("vit" in mname or "eva" in mname or "siglip" in mname):
                kw_g["img_size"] = eff_hw
            mg = timm.create_model(mname, **kw_g)
            mg.load_state_dict(sd_cpu, strict=True)
            mg = mg.to(dev_g).eval()
            mt = torch.tensor(mean_tup, device=dev_g, dtype=torch.float32).view(1, 3, 1, 1)
            st = torch.tensor(std_tup, device=dev_g, dtype=torch.float32).view(1, 3, 1, 1)
            worker_out = {}
            for sname, items, dbytes, all_idx, is_b in splits:
                sub_idx = all_idx[gid::2]
                r2, a6 = _extract_split_on_device(mg, mt, st, items, dbytes, sub_idx, eff_hw, eff_bs, is_b, dev_g, amp_g, workers=0, split_name=f"{sname}:g{gid}", n_views=N_VIEWS)
                worker_out[sname] = (sub_idx, r2, a6)
            del mg
            torch.cuda.empty_cache()
            return worker_out

        with ThreadPoolExecutor(max_workers=2) as pool:
            res0, res1 = list(pool.map(_worker, [0, 1]))
        for sname, _items, _db, all_idx, _isb in splits:
            dim = res0[sname][1].shape[1]
            r2_full = np.empty((len(all_idx), dim), dtype=np.float16)
            a6_full = np.empty((len(all_idx), dim), dtype=np.float16)
            for sub_idx, r2, a6 in (res0[sname], res1[sname]):
                r2_full[sub_idx] = r2
                a6_full[sub_idx] = a6
            results[f"{sname}_raw2"] = r2_full
            results[f"{sname}_all4"] = a6_full
    else:
        m = base_model.to(DEVICE).eval()
        mt = torch.tensor(mean_tup, device=DEVICE, dtype=torch.float32).view(1, 3, 1, 1)
        st = torch.tensor(std_tup, device=DEVICE, dtype=torch.float32).view(1, 3, 1, 1)
        for sname, items, dbytes, all_idx, is_b in splits:
            r2, a6 = _extract_split_on_device(m, mt, st, items, dbytes, all_idx, eff_hw, eff_bs, is_b, DEVICE, AMP_DTYPE, workers=NUM_WORKERS, split_name=sname, n_views=N_VIEWS)
            results[f"{sname}_raw2"] = r2
            results[f"{sname}_all4"] = a6
        del m

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    atomic_savez_compressed(out_npz, model_name=np.array([mname]), **results)
    print(f"[DONE] {fspec.key} dim={results['tr_all4'].shape[1]} saved -> {out_npz} ({time.time()-t0:.1f}s)", flush=True)
    return out_npz

EXTRACTED_NPZS = {}
if RUN_STAGE1_FOUNDATION_PROBES:
    for fspec in FOUND_SPECS:
        try:
            EXTRACTED_NPZS[fspec.key] = extract_backbone_embeddings(fspec)
        except Exception as _exc:
            print(f"[BACKBONE SKIP WARNING] {fspec.key} failed ({_exc}); continuing with remaining backbones...", flush=True)
    assert len(EXTRACTED_NPZS) >= 4, f"Too few foundation backbones succeeded ({len(EXTRACTED_NPZS)}): {list(EXTRACTED_NPZS)}"


In [ ]:
# Cell 5: Coarse Video-Cluster CV + Fold-Internal CL2N + 3 Upgraded Heads (LogReg, 5-Seed CosineResMLP, Nested PerBackbone-SLSQP)
import gc, math, os, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import minimize
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from torch.utils.data import DataLoader, TensorDataset

def _l2_norm_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, 1e-8, None)

def apply_cl2n_blocks(tr_blocks, ext_blocks, te_blocks, tr_mask: np.ndarray, va_mask: np.ndarray):
    """Pure-train fold-internal CL2N centering: subtract tr_mask mean per block and re-L2-normalize (zero test leakage)."""
    tr_c, va_c, ex_c, te_c = [], [], [], []
    for b_tr, b_ex, b_te in zip(tr_blocks, ext_blocks, te_blocks):
        mu = b_tr[tr_mask].mean(axis=0, keepdims=True)
        tr_c.append(_l2_norm_np(b_tr[tr_mask] - mu))
        va_c.append(_l2_norm_np(b_tr[va_mask] - mu))
        ex_c.append(_l2_norm_np(b_ex[va_mask] - mu))
        te_c.append(_l2_norm_np(b_te - mu))
    return tr_c, va_c, ex_c, te_c

def build_coarse_video_clusters(X_emb: np.ndarray, y_lbl: np.ndarray, n_per_class: int = 60, seed: int = 42) -> np.ndarray:
    """Diagnostic Leave-Video-Clusters-Out proxy inside train set (100% pure-train, zero test rows)."""
    groups = np.empty(len(y_lbl), dtype=np.int32)
    offset = 0
    for c in range(5):
        idx = np.where(y_lbl == c)[0]
        k_c = min(n_per_class, max(2, len(idx) // 25))
        km = MiniBatchKMeans(n_clusters=k_c, random_state=seed + c, batch_size=1024, n_init=3)
        groups[idx] = offset + km.fit_predict(X_emb[idx])
        offset += k_c
    return groups

class CosineResMLP(nn.Module):
    def __init__(self, in_dim: int, block_slices, hidden: int = 768, drop: float = 0.50, block_drop: float = 0.22):
        super().__init__()
        self.block_slices = tuple(block_slices)
        self.block_drop = float(block_drop)
        self.fc1 = nn.Linear(in_dim, hidden)
        self.ln1 = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(drop)
        self.res_fc = nn.Linear(hidden, hidden)
        self.ln2 = nn.LayerNorm(hidden)
        self.w = nn.Parameter(torch.randn(5, hidden) * 0.02)
        self.scale = nn.Parameter(torch.tensor(16.0))

    def _apply_block_drop(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or len(self.block_slices) <= 1 or self.block_drop <= 0.0:
            return x
        bsz = x.shape[0]
        n_blk = len(self.block_slices)
        keep = (torch.rand((bsz, n_blk), device=x.device) >= self.block_drop).float()
        none_kept = (keep.sum(dim=1, keepdim=True) == 0)
        if none_kept.any():
            rand_blk = F.one_hot(torch.randint(0, n_blk, (bsz,), device=x.device), num_classes=n_blk).float()
            keep = torch.where(none_kept, rand_blk, keep)
        scale = float(n_blk) / keep.sum(dim=1, keepdim=True)
        out = x.clone()
        for bi, (s, e) in enumerate(self.block_slices):
            out[:, s:e] = out[:, s:e] * (keep[:, bi : bi + 1] * scale)
        return out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self._apply_block_drop(x)
        h = F.gelu(self.ln1(self.fc1(x)))
        h = h + self.drop(F.gelu(self.ln2(self.res_fc(self.drop(h)))))
        z = F.normalize(h, p=2, dim=-1)
        w = F.normalize(self.w, p=2, dim=-1)
        return self.scale.clamp(4.0, 40.0) * (z @ w.T)

def train_cos_res_mlp_fold(X_tr, y_tr, X_eval_list, block_slices, seed: int, device: torch.device, epochs: int = 20):
    """Train CosineResMLP (hidden=768, Beta(0.4,0.4) Feature Mixup + exact tau=1.0 Logit-Adjusted CE); returns uniform-prior probs."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    pi_tr = np.bincount(y_tr, minlength=5).astype(np.float32) / len(y_tr)
    log_pi = torch.tensor(np.log(np.clip(pi_tr, 1e-4, 1.0)), device=device, dtype=torch.float32)

    ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr.astype(np.int64)))
    dl = DataLoader(ds, batch_size=min(256, len(ds)), shuffle=True, drop_last=False)
    model = CosineResMLP(X_tr.shape[1], block_slices, hidden=768, drop=0.50, block_drop=0.22).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1.1e-3, weight_decay=2.5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs * len(dl)), eta_min=1e-5)

    for _ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            xb = xb + torch.randn_like(xb) * 0.020
            yb_oh = F.one_hot(yb, num_classes=5).float()
            yb_oh = yb_oh * 0.98 + 0.02 / 5.0
            if np.random.random() < 0.50 and len(xb) > 1:
                lam = float(np.random.beta(0.40, 0.40))
                perm = torch.randperm(len(xb), device=device)
                xb = lam * xb + (1.0 - lam) * xb[perm]
                yb_oh = lam * yb_oh + (1.0 - lam) * yb_oh[perm]
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            lsm = F.log_softmax(logits + log_pi.unsqueeze(0), dim=1)
            loss = -(yb_oh * lsm).sum(dim=1).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
            sched.step()

    model.eval()
    outs = []
    with torch.no_grad():
        for Xe in X_eval_list:
            Xt = torch.from_numpy(Xe)
            probs = []
            for s in range(0, len(Xt), 1024):
                lg = model(Xt[s : s + 1024].to(device))
                probs.append(F.softmax(lg, dim=1).cpu().numpy())
            outs.append(np.concatenate(probs, axis=0))
    return outs

def unif_to_train_prior(p_unif: np.ndarray, pi_vec: np.ndarray) -> np.ndarray:
    """Convert balanced/uniform-prior probs P(x|c) -> train-prior posterior P_train(c|x) for exact fusion parity."""
    p = np.clip(p_unif, 1e-7, 1.0) * pi_vec[None, :]
    return p / p.sum(axis=1, keepdims=True)

def train_to_unif_prior(p_train: np.ndarray, pi_vec: np.ndarray) -> np.ndarray:
    """Convert train-prior posterior -> balanced/uniform-prior probs."""
    p = np.clip(p_train, 1e-7, 1.0) / np.clip(pi_vec[None, :], 1e-7, 1.0)
    return p / p.sum(axis=1, keepdims=True)

def fit_slsqp_simplex(p_list, y_lbl, weights_row, max_w: float = 0.80):
    m = len(p_list)
    lps = [np.log(np.clip(p, 1e-7, 1.0)) for p in p_list]
    wt = weights_row / np.mean(weights_row)
    idx = np.arange(len(y_lbl))
    def obj(w):
        w = np.maximum(w, 0.0); w /= (w.sum() + 1e-12)
        lp = sum(wi * lp_i for wi, lp_i in zip(w, lps))
        lp -= lp.max(axis=1, keepdims=True)
        ep = np.exp(lp); pr = ep / ep.sum(axis=1, keepdims=True)
        return float(-np.mean(wt * np.log(np.clip(pr[idx, y_lbl], 1e-12, 1.0))))
    res = minimize(obj, np.ones(m) / m, bounds=[(0.0, float(max_w))] * m, constraints={"type": "eq", "fun": lambda w: float(np.sum(w)) - 1.0}, method="SLSQP")
    w = np.maximum(res.x, 0.0); return w / w.sum()

def run_probe_heads_5fold(tr_blocks, ext_blocks, te_blocks, y_lbl, fold_arr, block_slices, mlp_seeds=None, epochs=20, best_C=1.0, device=DEVICE):
    if mlp_seeds is None:
        mlp_seeds = (42, 1024, 2025, 3141, 7777) if GPU_VRAM_GB >= 20.0 else (42, 1024, 2025)
    n_tr, n_te = len(tr_blocks[0]), len(te_blocks[0])

    oof_lr_u, oof_mlp_u, oof_pb_u = (np.zeros((n_tr, 5), dtype=np.float64) for _ in range(3))
    ext_lr_u, ext_mlp_u, ext_pb_u = (np.zeros((n_tr, 5), dtype=np.float64) for _ in range(3))
    te_lr_u, te_mlp_u, te_pb_u = (np.zeros((n_te, 5), dtype=np.float64) for _ in range(3))

    oof_lr_t, oof_mlp_t, oof_pb_t = (np.zeros((n_tr, 5), dtype=np.float64) for _ in range(3))
    ext_lr_t, ext_mlp_t, ext_pb_t = (np.zeros((n_tr, 5), dtype=np.float64) for _ in range(3))
    te_lr_t, te_mlp_t, te_pb_t = (np.zeros((n_te, 5), dtype=np.float64) for _ in range(3))
    bb_weights_folds = []

    for k in range(5):
        tr_m, va_m = (fold_arr != k), (fold_arr == k)
        pi_k = np.bincount(y_lbl[tr_m], minlength=5).astype(np.float64) / tr_m.sum()
        y_trk = y_lbl[tr_m]

        # Pure-train fold-internal CL2N centering per backbone
        tr_c_list, va_c_list, ex_c_list, te_c_list = apply_cl2n_blocks(tr_blocks, ext_blocks, te_blocks, tr_m, va_m)
        scale_b = math.sqrt(max(1, len(tr_c_list)))
        X_tr_k = np.concatenate(tr_c_list, axis=1).astype(np.float32) / scale_b
        X_va_k = np.concatenate(va_c_list, axis=1).astype(np.float32) / scale_b
        X_ex_k = np.concatenate(ex_c_list, axis=1).astype(np.float32) / scale_b
        X_te_k = np.concatenate(te_c_list, axis=1).astype(np.float32) / scale_b

        # Head 1: CL2N-centered Concatenated Balanced LogReg
        clf = LogisticRegression(C=best_C, class_weight="balanced", max_iter=500, random_state=42 + k)
        clf.fit(X_tr_k, y_trk)
        p_va1, p_ex1, p_te1 = clf.predict_proba(X_va_k), clf.predict_proba(X_ex_k), clf.predict_proba(X_te_k)
        oof_lr_u[va_m], ext_lr_u[va_m] = p_va1, p_ex1
        te_lr_u += p_te1 / 5.0
        oof_lr_t[va_m], ext_lr_t[va_m] = unif_to_train_prior(p_va1, pi_k), unif_to_train_prior(p_ex1, pi_k)
        te_lr_t += unif_to_train_prior(p_te1, pi_k) / 5.0

        # Head 2: 5-Seed CosineResMLP (BlockDrop + Beta(0.4,0.4) Mixup + tau=1.0 Logit-Adj CE)
        p_va2 = np.zeros((va_m.sum(), 5), dtype=np.float64)
        p_ex2 = np.zeros((va_m.sum(), 5), dtype=np.float64)
        p_te2 = np.zeros((n_te, 5), dtype=np.float64)
        for s_off in mlp_seeds:
            ov, oe, ot = train_cos_res_mlp_fold(X_tr_k, y_trk, [X_va_k, X_ex_k, X_te_k], block_slices, seed=s_off + k * 37, device=device, epochs=epochs)
            p_va2 += ov / len(mlp_seeds); p_ex2 += oe / len(mlp_seeds); p_te2 += ot / len(mlp_seeds)
        oof_mlp_u[va_m], ext_mlp_u[va_m] = p_va2, p_ex2
        te_mlp_u += p_te2 / 5.0
        oof_mlp_t[va_m], ext_mlp_t[va_m] = unif_to_train_prior(p_va2, pi_k), unif_to_train_prior(p_ex2, pi_k)
        te_mlp_t += unif_to_train_prior(p_te2, pi_k) / 5.0

        # Head 3: Nested Per-Backbone LogReg -> Inner 3-Fold SLSQP Simplex Fusion
        nb_len = len(tr_c_list)
        skf_in = StratifiedKFold(n_splits=3, shuffle=True, random_state=17 + k)
        inner_oof_t = [np.zeros((tr_m.sum(), 5), dtype=np.float64) for _ in range(nb_len)]
        va_pb_t, ex_pb_t, te_pb_t_list = [], [], []
        for bi in range(nb_len):
            xb_tr = tr_c_list[bi]
            for tr_i, va_i in skf_in.split(xb_tr, y_trk):
                pi_in = np.bincount(y_trk[tr_i], minlength=5).astype(np.float64) / len(tr_i)
                c_in = LogisticRegression(C=best_C, class_weight="balanced", max_iter=250, tol=1e-3, random_state=99 + k * 13 + bi)
                c_in.fit(xb_tr[tr_i], y_trk[tr_i])
                inner_oof_t[bi][va_i] = unif_to_train_prior(c_in.predict_proba(xb_tr[va_i]), pi_in)
            cb = LogisticRegression(C=best_C, class_weight="balanced", max_iter=500, random_state=99 + k * 13 + bi)
            cb.fit(xb_tr, y_trk)
            va_pb_t.append(unif_to_train_prior(cb.predict_proba(va_c_list[bi]), pi_k))
            ex_pb_t.append(unif_to_train_prior(cb.predict_proba(ex_c_list[bi]), pi_k))
            te_pb_t_list.append(unif_to_train_prior(cb.predict_proba(te_c_list[bi]), pi_k))

        w_bb_k = fit_slsqp_simplex(inner_oof_t, y_trk, np.ones(len(y_trk)), max_w=0.70)
        bb_weights_folds.append(w_bb_k)
        def _combine_bb(ps_list):
            lp = sum(wi * np.log(np.clip(pi, 1e-7, 1.0)) for wi, pi in zip(w_bb_k, ps_list))
            lp -= lp.max(axis=1, keepdims=True); ep = np.exp(lp); return ep / ep.sum(axis=1, keepdims=True)
        p_va3_t = _combine_bb(va_pb_t)
        p_ex3_t = _combine_bb(ex_pb_t)
        p_te3_t = _combine_bb(te_pb_t_list)
        oof_pb_t[va_m], ext_pb_t[va_m] = p_va3_t, p_ex3_t
        te_pb_t += p_te3_t / 5.0
        oof_pb_u[va_m], ext_pb_u[va_m] = train_to_unif_prior(p_va3_t, pi_k), train_to_unif_prior(p_ex3_t, pi_k)
        te_pb_u += train_to_unif_prior(p_te3_t, pi_k) / 5.0

    oof_t = np.zeros((n_tr, 5), dtype=np.float64)
    ext_t = np.zeros((n_tr, 5), dtype=np.float64)
    fold_ws = []
    for k in range(5):
        tr_m, va_m = (fold_arr != k), (fold_arr == k)
        w_k = fit_slsqp_simplex([oof_lr_t[tr_m], oof_mlp_t[tr_m], oof_pb_t[tr_m]], y_lbl[tr_m], np.ones(tr_m.sum()))
        fold_ws.append(w_k)
        def _gp(ps):
            lp = sum(wi * np.log(np.clip(pi, 1e-7, 1.0)) for wi, pi in zip(w_k, ps))
            lp -= lp.max(1, keepdims=True); ep = np.exp(lp); return ep / ep.sum(1, keepdims=True)
        oof_t[va_m] = _gp([oof_lr_t[va_m], oof_mlp_t[va_m], oof_pb_t[va_m]])
        ext_t[va_m] = _gp([ext_lr_t[va_m], ext_mlp_t[va_m], ext_pb_t[va_m]])
    w_mean = np.mean(fold_ws, axis=0)
    lp_te = sum(wi * np.log(np.clip(pi, 1e-7, 1.0)) for wi, pi in zip(w_mean, [te_lr_t, te_mlp_t, te_pb_t]))
    lp_te -= lp_te.max(1, keepdims=True); ep_te = np.exp(lp_te); te_t = ep_te / ep_te.sum(1, keepdims=True)

    oof_u = oof_lr_u * w_mean[0] + oof_mlp_u * w_mean[1] + oof_pb_u * w_mean[2]
    ext_u = ext_lr_u * w_mean[0] + ext_mlp_u * w_mean[1] + ext_pb_u * w_mean[2]
    bb_w_mean = np.round(np.mean(bb_weights_folds, axis=0), 4)
    print(f"  Learned PerBackbone-SLSQP weights: {bb_w_mean.tolist()}")
    return oof_t.astype(np.float32), ext_t.astype(np.float32), te_t.astype(np.float32), oof_u, ext_u, np.round(w_mean, 4)

def save_csv_preds(out_path: str, df_meta: pd.DataFrame, probs: np.ndarray, folds_col=None):
    fnames = df_meta["file_name"] if "file_name" in df_meta.columns else (df_meta["id"].astype(str) + ".jpg")
    out = pd.DataFrame({"id": df_meta["id"].values, "file_name": fnames.values})
    if "label" in df_meta.columns and df_meta["label"].notna().all():
        out["label"] = df_meta["label"].astype(int).values
    if folds_col is not None:
        out["fold"] = folds_col.astype(int)
    for c in range(5):
        out[f"prob_{c}"] = np.round(probs[:, c].astype(np.float64), 7)
    out.to_csv(out_path, index=False)

if RUN_STAGE1_FOUNDATION_PROBES and EXTRACTED_NPZS:
    raw2_tr_blocks, raw2_ex_blocks, raw2_te_blocks = [], [], []
    all4_tr_blocks, all4_ex_blocks, all4_te_blocks = [], [], []
    loaded_keys, slices = [], []
    cur = 0
    for fspec in FOUND_SPECS:
        if fspec.key not in EXTRACTED_NPZS:
            continue
        with np.load(EXTRACTED_NPZS[fspec.key]) as z:
            d = z["tr_all4"].shape[1]
            raw2_tr_blocks.append(_l2_norm_np(z["tr_raw2"])); raw2_ex_blocks.append(_l2_norm_np(z["ext_raw2"])); raw2_te_blocks.append(_l2_norm_np(z["te_raw2"]))
            all4_tr_blocks.append(_l2_norm_np(z["tr_all4"])); all4_ex_blocks.append(_l2_norm_np(z["ext_all4"])); all4_te_blocks.append(_l2_norm_np(z["te_all4"]))
            loaded_keys.append(fspec.key)
            slices.append((cur, cur + d))
            cur += d

    X_tr_a6 = np.concatenate(all4_tr_blocks, axis=1).astype(np.float32) / math.sqrt(len(all4_tr_blocks))
    print(f"\nCombined Multi-Foundation dim={cur} across {len(slices)} backbones: {list(zip(loaded_keys, slices))}")

    cluster_ids = build_coarse_video_clusters(X_tr_a6, y, n_per_class=60, seed=SEED)
    sgkf_coarse = list(StratifiedGroupKFold(n_splits=5).split(X_tr_a6, y, cluster_ids))
    del X_tr_a6; gc.collect()
    pi_all = np.bincount(y, minlength=5) / len(y)
    sw_all = 1.0 / pi_all[y]

    print("\n--- Per-Backbone Coarse Video-Cluster Diagnostic (C=1.0, CL2N: raw2 vs all6 + decorrupt_v4 + crop82) ---")
    for bkey, tr_r2_b, tr_a6_b in zip(loaded_keys, raw2_tr_blocks, all4_tr_blocks):
        oof_b_r2 = np.zeros((len(y), 5)); oof_b_a6 = np.zeros((len(y), 5))
        for tr_i, va_i in sgkf_coarse:
            mu_r = tr_r2_b[tr_i].mean(axis=0, keepdims=True)
            mu_a = tr_a6_b[tr_i].mean(axis=0, keepdims=True)
            oof_b_r2[va_i] = LogisticRegression(C=1.0, class_weight="balanced", max_iter=350, random_state=42).fit(_l2_norm_np(tr_r2_b[tr_i] - mu_r), y[tr_i]).predict_proba(_l2_norm_np(tr_r2_b[va_i] - mu_r))
            oof_b_a6[va_i] = LogisticRegression(C=1.0, class_weight="balanced", max_iter=350, random_state=42).fit(_l2_norm_np(tr_a6_b[tr_i] - mu_a), y[tr_i]).predict_proba(_l2_norm_np(tr_a6_b[va_i] - mu_a))
        fb_r2 = f1_score(y, oof_b_r2.argmax(1), average="macro", sample_weight=sw_all)
        fb_a6 = f1_score(y, oof_b_a6.argmax(1), average="macro", sample_weight=sw_all)
        print(f"  [{bkey:15s}] dim={tr_a6_b.shape[1]:4d} | raw2={fb_r2:.5f} -> all6={fb_a6:.5f} (delta={fb_a6-fb_r2:+.5f})")

    coarse_cl2n_cache = []
    scale_nb = math.sqrt(len(all4_tr_blocks))
    for tr_i, va_i in sgkf_coarse:
        m_tr = np.zeros(len(y), dtype=bool); m_tr[tr_i] = True
        m_va = np.zeros(len(y), dtype=bool); m_va[va_i] = True
        tr_r, va_r, _, _ = apply_cl2n_blocks(raw2_tr_blocks, raw2_ex_blocks, raw2_te_blocks, m_tr, m_va)
        tr_a, va_a, _, _ = apply_cl2n_blocks(all4_tr_blocks, all4_ex_blocks, all4_te_blocks, m_tr, m_va)
        coarse_cl2n_cache.append((
            tr_i, va_i,
            np.concatenate(tr_r, axis=1).astype(np.float32) / scale_nb,
            np.concatenate(va_r, axis=1).astype(np.float32) / scale_nb,
            np.concatenate(tr_a, axis=1).astype(np.float32) / scale_nb,
            np.concatenate(va_a, axis=1).astype(np.float32) / scale_nb,
        ))

    best_C, best_coarse_f1 = 1.0, -1.0
    print("\n--- Combined CL2N Coarse Video-Cluster Leave-Scene-Out CV (raw2 vs 6-view) ---")
    for C_val in [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]:
        oof_c_r2 = np.zeros((len(y), 5)); oof_c_a6 = np.zeros((len(y), 5))
        for tr_i, va_i, xtr_r, xva_r, xtr_a, xva_a in coarse_cl2n_cache:
            m_r = LogisticRegression(C=C_val, class_weight="balanced", max_iter=400, random_state=42).fit(xtr_r, y[tr_i])
            m_a = LogisticRegression(C=C_val, class_weight="balanced", max_iter=400, random_state=42).fit(xtr_a, y[tr_i])
            oof_c_r2[va_i] = m_r.predict_proba(xva_r)
            oof_c_a6[va_i] = m_a.predict_proba(xva_a)
        f_r2 = f1_score(y, oof_c_r2.argmax(1), average="macro", sample_weight=sw_all)
        f_a6 = f1_score(y, oof_c_a6.argmax(1), average="macro", sample_weight=sw_all)
        print(f"  C={C_val:4.2f} | CoarseCluster UnifF1: raw2={f_r2:.5f} vs all6(+decorrupt_v4+crop82)={f_a6:.5f} (delta={f_a6-f_r2:+.5f})")
        if f_a6 > best_coarse_f1:
            best_coarse_f1, best_C = f_a6, C_val
    del coarse_cl2n_cache; gc.collect()

    print(f"\n--- Training Production 5-Fold SGKF FoundVLM CL2N Heads (best_C={best_C}) ---")
    oof_orig2, _, te_orig2, oof_o2_u, _, w_r2 = run_probe_heads_5fold(
        raw2_tr_blocks, raw2_ex_blocks, raw2_te_blocks, y, folds, slices, best_C=best_C, device=DEVICE
    )
    oof_w6, ext_w6, te_w6, oof_w6_u, ext_w6_u, w_a6 = run_probe_heads_5fold(
        all4_tr_blocks, all4_ex_blocks, all4_te_blocks, y, folds, slices, best_C=best_C, device=DEVICE
    )

    f_cl_t = f1_score(y, oof_w6.argmax(1), average="macro")
    f_cl_u = f1_score(y, oof_w6_u.argmax(1), average="macro", sample_weight=sw_all)
    f_ex_u = f1_score(y, ext_w6_u.argmax(1), average="macro", sample_weight=sw_all)
    ll_cl = log_loss(y, np.clip(oof_w6, 1e-7, 1.0), labels=[0, 1, 2, 3, 4])
    print(f"\n=== [foundvlm] 5-FOLD COMPLETE ({len(loaded_keys)} backbones, CL2N + Nested PerBackbone-SLSQP) ===")
    print(f"  Head Weights [LogReg, CosineResMLP, PerBackboneSLSQP]: raw2={w_r2.tolist()} | all6={w_a6.tolist()}")
    print(f"  SGKF OOF: TrainPriorF1={f_cl_t:.5f} | UnifPriorF1={f_cl_u:.5f} | OOF_LL={ll_cl:.5f} | ExtHard UnifF1={f_ex_u:.5f} | CoarseCluster UnifF1={best_coarse_f1:.5f}")
    print(f"  Test predicted dist (train-prior argmax): {pd.Series(te_w6.argmax(1)).value_counts().sort_index().to_dict()}")

    save_csv_preds(os.path.join(OUT_DIR, "task4-foundvlm-hard6-oof.csv"), tr_df, oof_w6, folds)
    save_csv_preds(os.path.join(OUT_DIR, "task4-foundvlm-hard6-test-predictions.csv"), te_df, te_w6)
    save_csv_preds(os.path.join(OUT_DIR, "task4-foundvlm-orig2-oof.csv"), tr_df, oof_orig2, folds)
    save_csv_preds(os.path.join(OUT_DIR, "task4-foundvlm-orig2-test-predictions.csv"), te_df, te_orig2)
    atomic_savez_compressed(
        os.path.join(OUT_DIR, "task4-foundvlm-sgkf-hard-bundle.npz"),
        folds=folds, oof_orig2=oof_orig2, oof_w6=oof_w6, ext_w6=ext_w6, te_orig2=te_orig2, te_w6=te_w6,
    )
    fzip = os.path.join(OUT_DIR, "t4_foundvlm.zip")
    with zipfile.ZipFile(fzip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fn in (
            "task4-foundvlm-hard6-oof.csv",
            "task4-foundvlm-hard6-test-predictions.csv",
            "task4-foundvlm-orig2-oof.csv",
            "task4-foundvlm-orig2-test-predictions.csv",
            "task4-foundvlm-sgkf-hard-bundle.npz",
        ):
            zf.write(os.path.join(OUT_DIR, fn), fn)
    print(f"[READY] Packaged FoundVLM bundle -> {fzip}", flush=True)
    del raw2_tr_blocks, raw2_ex_blocks, raw2_te_blocks, all4_tr_blocks, all4_ex_blocks, all4_te_blocks
    gc.collect()


In [ ]:
# Cell 6: Stage 2 — Dual-Model L2-SP Clean Low-LR Fine-Tuning @ (224, 448) + 6-View GPU TTA (P_NIGHTMARE=0.0)
import gc, io, math, os, random, shutil, threading, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset

MODEL_INIT_LOCK = threading.Lock()

if "_center_crop_resize" not in globals():
    def _center_crop_resize(x: torch.Tensor, frac: float = 0.82) -> torch.Tensor:
        _, _, h, w = x.shape
        ch, cw = max(8, int(round(h * frac))), max(8, int(round(w * frac)))
        y0, x0 = (h - ch) // 2, (w - cw) // 2
        crop = x[:, :, y0 : y0 + ch, x0 : x0 + cw]
        return F.interpolate(crop, size=(h, w), mode="bicubic", align_corners=False).clamp(0.0, 1.0)

@dataclass(frozen=True)
class CleanMTSpec:
    key: str
    timm_name: str
    img_hw: Tuple[int, int]
    freeze_vit_blocks: int
    base_lr_head: float
    backbone_mult: float
    batch_t4: int
    eval_bs: int

CLEAN_SPECS = {
    "clean_siglip": CleanMTSpec("clean_siglip", "vit_so400m_patch14_siglip_384", (224, 448) if HIGH_RES_A100 else (196, 392), 12, 3.0e-4, 0.08, 12, 64),
    "clean_dinov3l": CleanMTSpec("clean_dinov3l", "vit_large_patch16_dinov3.lvd1689m", (224, 448) if HIGH_RES_A100 else (192, 384), 12, 3.0e-4, 0.08, 16, 96),
}

def apply_l2sp_step(bb_init, cur_bb_lr: float, sp_wd: float):
    """Decoupled L2-SP decay pulling unfrozen backbone weights gently toward pretrained theta_0."""
    if sp_wd <= 0.0 or cur_bb_lr <= 0.0 or not bb_init:
        return
    alpha = float(cur_bb_lr) * float(sp_wd)
    with torch.no_grad():
        for p, p0 in bb_init:
            p.data.add_(p0.to(dtype=p.dtype) - p.data, alpha=alpha)

class CleanTrainDS(Dataset):
    """50% raw + 50% precomputed decorrupt_v4 + random hflip (P_NIGHTMARE=0.0 -> zero double corruption!)."""
    def __init__(self, raw_items, dec_bytes, idx, labels, img_hw, seed=42):
        self.raw_items, self.dec_bytes, self.idx, self.labels = raw_items, dec_bytes, idx, labels
        self.wh = (img_hw[1], img_hw[0])
        self.seed = seed
        self._rng = None

    def _get_rng(self):
        if self._rng is None:
            info = torch.utils.data.get_worker_info()
            self._rng = np.random.default_rng(self.seed + (info.id if info else 0) * 97)
        return self._rng

    def __len__(self): return len(self.idx)

    def __getitem__(self, pos: int):
        rng = self._get_rng()
        ri = int(self.idx[pos])
        src = self.dec_bytes[ri] if rng.random() < 0.50 else self.raw_items[ri]
        im = Image.open(io.BytesIO(src)).convert("RGB")
        if rng.random() < 0.50:
            im = im.transpose(Image.FLIP_LEFT_RIGHT)
        if im.size != self.wh:
            im = im.resize(self.wh, Image.BICUBIC)
        arr = np.asarray(im, dtype=np.uint8)
        return torch.from_numpy(np.array(arr, copy=True)).permute(2, 0, 1).contiguous(), int(self.labels[ri])

class CleanEvalDS(Dataset):
    """Yield [raw, precomputed_decorrupt_v4] pair per item for clean 6-view GPU TTA."""
    def __init__(self, raw_items, dec_bytes, idx, img_hw, is_bytes=True):
        self.raw_items = raw_items
        self.dec_bytes = dec_bytes
        self.idx = idx if idx is not None else np.arange(len(raw_items))
        self.wh = (img_hw[1], img_hw[0])
        self.is_bytes = is_bytes

    def __len__(self): return len(self.idx)

    def __getitem__(self, pos: int):
        ri = int(self.idx[pos])
        obj = self.raw_items[ri]
        im_r = Image.open(io.BytesIO(obj)).convert("RGB") if self.is_bytes else Image.fromarray(obj)
        im_d = Image.open(io.BytesIO(self.dec_bytes[ri])).convert("RGB")
        if im_r.size != self.wh: im_r = im_r.resize(self.wh, Image.BICUBIC)
        if im_d.size != self.wh: im_d = im_d.resize(self.wh, Image.BICUBIC)
        pair = np.stack([np.asarray(im_r, dtype=np.uint8), np.asarray(im_d, dtype=np.uint8)], axis=0)
        return torch.from_numpy(pair).permute(0, 3, 1, 2).contiguous()

def run_clean_finetune_stage2(cspec: CleanMTSpec):
    cache_dir = os.path.join(OUT_DIR, f"folds_{cspec.key}")
    os.makedirs(cache_dir, exist_ok=True)
    folds_run = STAGE2_FOLDS_TO_RUN if STAGE2_FOLDS_TO_RUN is not None else [0, 1, 2, 3, 4]

    if GPU_VRAM_GB >= 70.0:
        train_bs, eval_bs, use_grad_ckpt = 48, 128, False
    elif GPU_VRAM_GB >= 36.0:
        train_bs, eval_bs, use_grad_ckpt = 24, 96, False
    elif GPU_VRAM_GB >= 20.0:
        train_bs, eval_bs, use_grad_ckpt = 16, 64, True
    else:
        train_bs, eval_bs, use_grad_ckpt = cspec.batch_t4, 32, True

    dl_workers = 0 if (ENABLE_DUAL_GPU_PARALLEL and N_GPUS >= 2) else NUM_WORKERS
    pin_mem = (DEVICE.type == "cuda" and dl_workers > 0)

    def _train_fold(k: int, device: torch.device):
        fold_npz = os.path.join(cache_dir, f"fold{k}.npz")
        if os.path.isfile(fold_npz):
            try:
                with np.load(fold_npz) as z:
                    res_c = {key: z[key].copy() for key in ("va_orig2", "va_w6", "ext_w6", "te_orig2", "te_w6")}
                    print(f"[{cspec.key}] Fold {k} loaded from cache ({fold_npz})")
                    return res_c
            except Exception:
                pass
        if device.type == "cuda": torch.cuda.set_device(device)
        amp_d, use_sc = select_amp_dtype(device)
        random.seed(SEED + k); np.random.seed(SEED + k); torch.manual_seed(SEED + k)
        kw = {"pretrained": True, "num_classes": 5}
        with MODEL_INIT_LOCK:
            try:
                model = timm.create_model(cspec.timm_name, img_size=cspec.img_hw, **kw)
                eff_hw = cspec.img_hw
            except TypeError:
                model = timm.create_model(cspec.timm_name, **kw)
                eff_hw = tuple(timm.data.resolve_model_data_config(model)["input_size"][1:])
        if hasattr(model, "set_grad_checkpointing") and (use_grad_ckpt or not DISABLE_GRAD_CKPT_HIGH_VRAM):
            model.set_grad_checkpointing(True)
        for nm, p in model.named_parameters():
            if nm.startswith("blocks."):
                ps = nm.split(".")
                if len(ps) > 1 and ps[1].isdigit() and int(ps[1]) < cspec.freeze_vit_blocks:
                    p.requires_grad_(False)
        cfg = timm.data.resolve_model_data_config(model)
        mt = torch.tensor(cfg["mean"], device=device, dtype=torch.float32).view(1, 3, 1, 1)
        st = torch.tensor(cfg["std"], device=device, dtype=torch.float32).view(1, 3, 1, 1)
        model = model.to(device)

        hd, bb = [], []
        for nm, p in model.named_parameters():
            if not p.requires_grad: continue
            (hd if (nm.startswith("head") or nm.startswith("fc") or nm.startswith("attn_pool")) else bb).append(p)
        bb_init = [(p, p.detach().clone()) for p in bb] if SP_WEIGHT_DECAY > 0.0 else []
        opt = torch.optim.AdamW([
            {"params": hd, "lr": cspec.base_lr_head, "weight_decay": 2e-2},
            {"params": bb, "lr": cspec.base_lr_head * cspec.backbone_mult, "weight_decay": 0.0 if SP_WEIGHT_DECAY > 0.0 else 2e-2},
        ])
        tr_idx, va_idx = np.where(folds != k)[0], np.where(folds == k)[0]
        pi_k = np.bincount(y[tr_idx], minlength=5).astype(np.float32) / len(tr_idx)
        log_pi_t = torch.tensor(np.log(np.clip(pi_k, 1e-4, 1.0)), device=device, dtype=torch.float32)
        dl = DataLoader(CleanTrainDS(tr_bytes, tr_dec_bytes, tr_idx, y, eff_hw, seed=SEED + k), batch_size=train_bs, shuffle=True, drop_last=True, num_workers=dl_workers, pin_memory=pin_mem)
        accum = max(1, math.ceil(48 / train_bs))
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, STAGE2_EPOCHS * math.ceil(len(dl) / accum)), eta_min=1e-6)
        scaler = torch.amp.GradScaler("cuda", enabled=use_sc) if device.type == "cuda" else None
        ema = {ek: ev.detach().clone() for ek, ev in model.state_dict().items()}
        print(f"[{cspec.key} F{k}] Start Stage-2 L2-SP clean train @ {eff_hw} | bs={train_bs}x{accum} eval_bs={eval_bs} ckpt={use_grad_ckpt} sp_wd={SP_WEIGHT_DECAY}", flush=True)

        for _ep in range(STAGE2_EPOCHS):
            ep_t0 = time.time()
            model.train(); opt.zero_grad(set_to_none=True)
            for si, (xb_u8, yb) in enumerate(dl):
                xb = ((xb_u8.to(device, non_blocking=pin_mem).float() / 255.0) - mt) / st
                yb = yb.to(device, non_blocking=pin_mem)
                with torch.amp.autocast(device.type, dtype=amp_d) if amp_d is not None else torch.enable_grad():
                    logits = model(xb)
                    loss = F.cross_entropy(logits.float() + log_pi_t, yb, label_smoothing=0.03) / accum
                if scaler is not None: scaler.scale(loss).backward()
                else: loss.backward()
                if ((si + 1) % accum == 0) or (si + 1 == len(dl)):
                    if scaler is not None:
                        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                        scaler.step(opt); scaler.update()
                    else:
                        nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step()
                    apply_l2sp_step(bb_init, float(opt.param_groups[1]["lr"]), SP_WEIGHT_DECAY)
                    sched.step(); opt.zero_grad(set_to_none=True)
                    with torch.no_grad():
                        for ek, ev in model.state_dict().items():
                            if ev.dtype.is_floating_point: ema[ek].mul_(0.996).add_(ev.detach(), alpha=0.004)
            print(f"[{cspec.key} F{k}] ep {_ep+1}/{STAGE2_EPOCHS} ({time.time()-ep_t0:.1f}s)", flush=True)

        model.load_state_dict(ema, strict=False); model.eval()
        if hasattr(model, "set_grad_checkpointing"): model.set_grad_checkpointing(False)

        def _eval_clean(items, dbytes, idx, is_b):
            eds = CleanEvalDS(items, dbytes, idx, eff_hw, is_bytes=is_b)
            edl = DataLoader(eds, batch_size=eval_bs, shuffle=False, num_workers=dl_workers, pin_memory=pin_mem)
            p_o2, p_a6 = [], []
            eff_v_eval = 4 if globals().get("N_VIEWS", 6) <= 4 else 6
            tta_w = (0.25, 0.25, 0.25, 0.25) if eff_v_eval == 4 else (0.20, 0.20, 0.20, 0.20, 0.10, 0.10)
            with torch.inference_mode():
                for pair_u8 in edl:
                    pf = pair_u8.to(device, non_blocking=pin_mem).float() / 255.0
                    xr, xd = pf[:, 0], pf[:, 1]
                    vs = [xr, torch.flip(xr, [-1]), xd, torch.flip(xd, [-1])]
                    if eff_v_eval == 6:
                        vs.extend([_center_crop_resize(xr, 0.82), _center_crop_resize(xd, 0.82)])
                    pr = []
                    for v in vs:
                        with torch.amp.autocast(device.type, dtype=amp_d) if amp_d is not None else torch.inference_mode():
                            lg = model((v - mt) / st)
                        pr.append(F.softmax(lg.float() + log_pi_t, dim=1).cpu().numpy())
                    o2 = np.exp(0.55 * np.log(np.clip(pr[0], 1e-7, 1.0)) + 0.45 * np.log(np.clip(pr[1], 1e-7, 1.0)))
                    o2 /= o2.sum(1, keepdims=True)
                    a6 = np.exp(sum(w_v * np.log(np.clip(p_v, 1e-7, 1.0)) for w_v, p_v in zip(tta_w, pr)))
                    a6 /= a6.sum(1, keepdims=True)
                    p_o2.append(o2); p_a6.append(a6)
            return np.concatenate(p_o2, 0).astype(np.float32), np.concatenate(p_a6, 0).astype(np.float32)

        vo2, va6 = _eval_clean(tr_bytes, tr_dec_bytes, va_idx, True)
        _, ea6 = _eval_clean(ext_all_u8, ext_dec_bytes, va_idx, False)
        to2, ta6 = _eval_clean(te_bytes, te_dec_bytes, None, True)
        res = {"va_orig2": vo2, "va_w6": va6, "ext_w6": ea6, "te_orig2": to2, "te_w6": ta6}
        atomic_savez_compressed(fold_npz, **res)
        print(f"[{cspec.key}] DONE Fold {k} | Val6View F1={f1_score(y[va_idx], va6.argmax(1), average='macro'):.5f} | ExtHard F1={f1_score(y[va_idx], ea6.argmax(1), average='macro'):.5f}", flush=True)
        del model, opt, sched, ema, bb_init
        gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
        return res

    fold_res = {}
    if ENABLE_DUAL_GPU_PARALLEL and N_GPUS >= 2 and DEVICE.type == "cuda":
        with ThreadPoolExecutor(max_workers=2) as pool:
            futs = [pool.submit(lambda g, fs: {fk: _train_fold(fk, torch.device(f"cuda:{g}")) for fk in fs}, gid, folds_run[gid::2]) for gid in range(2) if folds_run[gid::2]]
            for f in futs: fold_res.update(f.result())
    else:
        for fk in folds_run: fold_res[fk] = _train_fold(fk, DEVICE)

    if len(fold_res) == 5:
        oof_o2 = np.zeros((len(tr_df), 5), np.float32); oof_w6 = np.zeros_like(oof_o2); ext_w6 = np.zeros_like(oof_o2)
        te_o2 = np.zeros((len(te_df), 5), np.float32); te_w6 = np.zeros_like(te_o2)
        for k in range(5):
            va = np.where(folds == k)[0]
            oof_o2[va], oof_w6[va], ext_w6[va] = fold_res[k]["va_orig2"], fold_res[k]["va_w6"], fold_res[k]["ext_w6"]
            te_o2 += fold_res[k]["te_orig2"] / 5.0; te_w6 += fold_res[k]["te_w6"] / 5.0
        f_names = (
            f"task4-{cspec.key}-hard6-oof.csv",
            f"task4-{cspec.key}-hard6-test-predictions.csv",
            f"task4-{cspec.key}-orig2-oof.csv",
            f"task4-{cspec.key}-orig2-test-predictions.csv",
            f"task4-{cspec.key}-sgkf-hard-bundle.npz",
        )
        save_csv_preds(os.path.join(OUT_DIR, f_names[0]), tr_df, oof_w6, folds)
        save_csv_preds(os.path.join(OUT_DIR, f_names[1]), te_df, te_w6)
        save_csv_preds(os.path.join(OUT_DIR, f_names[2]), tr_df, oof_o2, folds)
        save_csv_preds(os.path.join(OUT_DIR, f_names[3]), te_df, te_o2)
        atomic_savez_compressed(os.path.join(OUT_DIR, f_names[4]), folds=folds, oof_orig2=oof_o2, oof_w6=oof_w6, ext_w6=ext_w6, te_orig2=te_o2, te_w6=te_w6)
        mzip = os.path.join(OUT_DIR, f"t4_{cspec.key}.zip")
        with zipfile.ZipFile(mzip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for fn in f_names:
                zf.write(os.path.join(OUT_DIR, fn), fn)
        if IS_COLAB and os.path.abspath(OUT_DIR) != "/content/t4_found_outputs":
            shutil.copy2(mzip, f"/content/t4_{cspec.key}.zip")
        print(f"[{cspec.key}] 5-FOLD COMPLETE! OOF F1={f1_score(y, oof_w6.argmax(1), average='macro'):.5f} | ExtHard F1={f1_score(y, ext_w6.argmax(1), average='macro'):.5f} -> {mzip}", flush=True)

if DO_STAGE2:
    for _mkey in STAGE2_MODEL_KEYS:
        run_clean_finetune_stage2(CLEAN_SPECS[_mkey])
else:
    print(f"Stage 2 Clean Fine-Tuning skipped (VRAM={GPU_VRAM_GB:.1f}GB < 36GB and RUN_STAGE2_CLEAN_FINETUNE={RUN_STAGE2_CLEAN_FINETUNE}).")


In [ ]:
# Cell 7: Final Bundle Verification, Combined Packaging -> t4_found_bundles.zip & Colab Download Copy
import glob, os, shutil, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, log_loss

bundle_files = sorted(glob.glob(os.path.join(OUT_DIR, "task4-*.csv")) + glob.glob(os.path.join(OUT_DIR, "task4-*.npz")))
assert bundle_files, f"No completed Task 4 bundles found in {OUT_DIR}!"

pi_all = np.bincount(y, minlength=5) / len(y)
sw_unif = 1.0 / pi_all[y]

zip_path = os.path.join(OUT_DIR, "t4_found_bundles.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fn in bundle_files:
        zf.write(fn, os.path.basename(fn))
        if fn.endswith("hard6-oof.csv"):
            df_o = pd.read_csv(fn)
            p_o = df_o[[f"prob_{i}" for i in range(5)]].to_numpy()
            p_o /= np.clip(p_o.sum(axis=1, keepdims=True), 1e-12, None)
            p_u = p_o / pi_all[None, :]
            f_tr = f1_score(y, p_o.argmax(1), average="macro")
            f_un = f1_score(y, p_u.argmax(1), average="macro", sample_weight=sw_unif)
            ll = log_loss(y, np.clip(p_o, 1e-7, 1.0), labels=[0, 1, 2, 3, 4])
            npz_p = fn.replace("-hard6-oof.csv", "-sgkf-hard-bundle.npz")
            ext_s = ""
            if os.path.isfile(npz_p):
                with np.load(npz_p) as z:
                    pe_u = z["ext_w6"] / pi_all[None, :]
                    ext_s = f" | ExtHard UnifF1={f1_score(y, pe_u.argmax(1), average='macro', sample_weight=sw_unif):.5f}"
            print(f"{os.path.basename(fn):40s} | TrF1={f_tr:.5f} | UnifF1={f_un:.5f} | LL={ll:.5f}{ext_s}")

print("Packaged all completed outputs ->", zip_path, f"({os.path.getsize(zip_path)/1024**2:.2f} MB)")
if IS_COLAB:
    if os.path.abspath(OUT_DIR) != "/content/t4_found_outputs":
        shutil.copy2(zip_path, "/content/t4_found_bundles.zip")
        for _zfn in ("t4_foundvlm.zip", "t4_clean_siglip.zip", "t4_clean_dinov3l.zip"):
            _zp = os.path.join(OUT_DIR, _zfn)
            if os.path.isfile(_zp):
                shutil.copy2(_zp, f"/content/{_zfn}")
        print(f"Colab Download Ready! Drive={zip_path} | Colab Sidebar=/content/t4_found_bundles.zip")
    else:
        print(f"Colab Download Ready! Sidebar={zip_path}")
